# 👁️ Third Eye: AI earpiece prototype (Colab version)

This notebook runs the **brain** of the Third Eye earpiece in Google Colab:

1. **Speech to text:** upload a lecture recording (audio or video) and get a time-stamped transcript.
2. **Third Eye camera:** if you upload a *video*, it finds each new slide or whiteboard, saves a picture, and reads the text on it.
3. **Ask the assistant:** type a question, or speak one through your browser mic, and hear the answer.
4. **Study sheet + reminders:** a one-page study sheet (Word + Markdown) and a calendar file with review reminders and homework deadlines.

Everything is saved to **Google Drive → My Drive → ThirdEye**, so it shows up on your phone. That's the "personal cloud" part of the invention.

### How to use
* Run the cells **in order**: click ▶ on each one, or use **Runtime → Run all** and answer the prompts.
* *Optional (faster):* **Runtime → Change runtime type → T4 GPU** before you start.
* *Optional (much smarter answers and study sheets):* add an AI key. Click the 🔑 **Secrets** icon in the left sidebar and add **one** of these, with *Notebook access* turned on:
  * `GEMINI_API_KEY`: **free**. Get it at [aistudio.google.com](https://aistudio.google.com) → *Get API key*.
  * `ANTHROPIC_API_KEY`: Claude, pay-as-you-go, from console.anthropic.com.

  Without a key, everything still works using simpler offline methods.

> Colab runs on Google's servers, so it can't use your webcam live. The laptop version (`main.py`) does that. Here you work with a recording, *after* the lecture.

## 1. Setup (takes about 1 minute)

In [ ]:
#@title ▶ Install everything { display-mode: "form" }
!pip install -q faster-whisper python-docx pytesseract gTTS
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1

# Write the project's Python files (same code as the laptop version in third_eye.zip)
import base64, os, sys, pathlib
FILES = {
 "config.py": "IiIiClNldHRpbmdzIGZvciB0aGUgVGhpcmQgRXllIGVhcnBpZWNlIHByb3RvdHlwZS4KQ2hhbmdlIHRoZXNlIHRvIGZpdCB5b3VyIGxhcHRvcCAvIGRlbW8uIE1vc3QgY2FuIGFsc28gYmUgc2V0IGZyb20gdGhlIGNvbW1hbmQgbGluZS4KIiIiCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCiMgLS0tLS0tLS0tLSBXaGVyZSBub3RlcyBnbyAoInBlcnNvbmFsIGNsb3VkIikgLS0tLS0tLS0tLQojIFBvaW50IHRoaXMgYXQgYSBmb2xkZXIgeW91ciBwaG9uZSBzeW5jcywgZS5nLgojICAgV2luZG93czogciJDOlxVc2Vyc1x5b3VcR29vZ2xlIERyaXZlXFRoaXJkRXllIgojICAgbWFjT1M6ICAgIn4vTGlicmFyeS9DbG91ZFN0b3JhZ2UvR29vZ2xlRHJpdmUteW91QGdtYWlsLmNvbS9NeSBEcml2ZS9UaGlyZEV5ZSIKIyAgIG9yIGEgRHJvcGJveCAvIE9uZURyaXZlIC8gaUNsb3VkIERyaXZlIC8gTmV4dGNsb3VkIGZvbGRlci4KU1lOQ19GT0xERVIgPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJUSElSRF9FWUVfU1lOQyIsICIuL1RoaXJkRXllTm90ZXMiKSkuZXhwYW5kdXNlcigpCgojIC0tLS0tLS0tLS0gRGFzaGJvYXJkIC0tLS0tLS0tLS0KREFTSEJPQVJEX1BPUlQgPSA4MDAwICAgICAgICAjIHRoZSBkYXNoYm9hcmQgb3BlbnMgYXQgaHR0cDovL2xvY2FsaG9zdDo4MDAwCgojIC0tLS0tLS0tLS0gU3BlZWNoIHRvIHRleHQgLS0tLS0tLS0tLQojIGZhc3Rlci13aGlzcGVyIG1vZGVsIHNpemVzOiAidGlueS5lbiIsICJiYXNlLmVuIiwgInNtYWxsLmVuIiwgIm1lZGl1bS5lbiIKIyB0aW55L2Jhc2UgYXJlIGZhc3QgZW5vdWdoIGZvciBtb3N0IGxhcHRvcHMgb24gQ1BVOyBzbWFsbCBpcyBtb3JlIGFjY3VyYXRlLgpXSElTUEVSX01PREVMID0gb3MuZW52aXJvbi5nZXQoIlRISVJEX0VZRV9XSElTUEVSIiwgImJhc2UuZW4iKQpTQU1QTEVfUkFURSA9IDE2MDAwICAgICAgICAgICMgV2hpc3BlciBleHBlY3RzIDE2IGtIeiBtb25vCkNIVU5LX1NFQ09ORFMgPSA4ICAgICAgICAgICAgIyBob3cgbXVjaCBsZWN0dXJlIGF1ZGlvIHRvIHRyYW5zY3JpYmUgYXQgYSB0aW1lClFVRVNUSU9OX1NFQ09ORFMgPSA2ICAgICAgICAgIyBob3cgbG9uZyBwdXNoLXRvLXRhbGsgbGlzdGVucyBmb3IgeW91ciBxdWVzdGlvbgpTSUxFTkNFX1JNUyA9IDAuMDA0ICAgICAgICAgICMgY2h1bmtzIHF1aWV0ZXIgdGhhbiB0aGlzIGFyZSBza2lwcGVkICh0dW5lIGZvciB5b3VyIG1pYykKCiMgLS0tLS0tLS0tLSBDYW1lcmEgKHRoZSBUaGlyZCBFeWUgY2FtZXJhKSAtLS0tLS0tLS0tCkNBTUVSQV9JTkRFWCA9IDAgICAgICAgICAgICAgIyAwID0gYnVpbHQtaW4gd2ViY2FtOyB0cnkgMSBmb3IgYSBVU0IgY2FtZXJhClNMSURFX0NIQU5HRV9USFJFU0hPTEQgPSAzICAgIyAlIG9mIHRoZSBwaWN0dXJlIHRoYXQgbXVzdCBjaGFuZ2UgdG8gY291bnQgYXMgYSBuZXcgc2xpZGUKU0xJREVfU1RBQkxFX0ZSQU1FUyA9IDE1ICAgICAjIGZyYW1lcyB0aGUgbmV3IHZpZXcgbXVzdCBzdGF5IHN0aWxsIGJlZm9yZSB3ZSBzbmFwc2hvdCBpdApNSU5fU0VDT05EU19CRVRXRUVOX1NMSURFUyA9IDUKCiMgLS0tLS0tLS0tLSBBSSBhc3Npc3RhbnQgLS0tLS0tLS0tLQojIFNldCBPTkUgb2YgdGhlc2UgZW52aXJvbm1lbnQgdmFyaWFibGVzIHRvIHR1cm4gb24gQUkgYW5zd2VycyBhbmQgQUkgc3R1ZHkgc2hlZXRzOgojICAgQU5USFJPUElDX0FQSV9LRVkgIC0+IENsYXVkZSAocGFpZCwgcGF5LWFzLXlvdS1nbykKIyAgIEdFTUlOSV9BUElfS0VZICAgICAtPiBHb29nbGUgR2VtaW5pIChoYXMgYSBGUkVFIHRpZXI6IGdldCBhIGtleSBhdCBhaXN0dWRpby5nb29nbGUuY29tKQojIElmIGJvdGggYXJlIHNldCwgQ2xhdWRlIGlzIHVzZWQuIFdpdGggbmVpdGhlciwgdGhlIHByb2dyYW0gc3RpbGwgd29ya3MgdXNpbmcgc2ltcGxlIG9mZmxpbmUgbWV0aG9kcy4KQU5USFJPUElDX0FQSV9LRVkgPSBvcy5lbnZpcm9uLmdldCgiQU5USFJPUElDX0FQSV9LRVkiLCAiIikKQUlfTU9ERUwgPSBvcy5lbnZpcm9uLmdldCgiVEhJUkRfRVlFX01PREVMIiwgImNsYXVkZS1zb25uZXQtNS01IikKR0VNSU5JX0FQSV9LRVkgPSBvcy5lbnZpcm9uLmdldCgiR0VNSU5JX0FQSV9LRVkiLCAiIikgb3Igb3MuZW52aXJvbi5nZXQoIkdPT0dMRV9BUElfS0VZIiwgIiIpCkdFTUlOSV9NT0RFTCA9IG9zLmVudmlyb24uZ2V0KCJUSElSRF9FWUVfR0VNSU5JX01PREVMIiwgIiIpICAgIyBvcHRpb25hbDogZm9yY2UgYSBtb2RlbCBmb3Igc3R1ZHkgc2hlZXRzCgoKZGVmIGFpX3Byb3ZpZGVyKCkgLT4gc3RyOgogICAgIiIiV2hpY2ggQUkgaXMgc3dpdGNoZWQgb246ICJDbGF1ZGUiLCAiR2VtaW5pIiBvciAiIiAob2ZmbGluZSkuIiIiCiAgICBpZiBBTlRIUk9QSUNfQVBJX0tFWToKICAgICAgICByZXR1cm4gIkNsYXVkZSIKICAgIGlmIEdFTUlOSV9BUElfS0VZOgogICAgICAgIHJldHVybiAiR2VtaW5pIgogICAgcmV0dXJuICIiCkNPTlRFWFRfTUlOVVRFUyA9IDUgICAgICAgICAgIyBob3cgbXVjaCByZWNlbnQgbGVjdHVyZSB0aGUgYXNzaXN0YW50ICJyZW1lbWJlcnMiIGZvciBsaXZlIHF1ZXN0aW9ucwoKIyAtLS0tLS0tLS0tIFN0dWR5IHNoZWV0IHJlbWluZGVycyAtLS0tLS0tLS0tClJFVklFV19EQVlTID0gWzEsIDMsIDddICAgICAgIyBzcGFjZWQtcmVwZXRpdGlvbiByZXZpZXcgcmVtaW5kZXJzIChkYXlzIGFmdGVyIGxlY3R1cmUpClJFTUlOREVSX0hPVVIgPSAxOCAgICAgICAgICAgIyByZW1pbmRlcnMgZmlyZSBhdCA2IHBtIGxvY2FsIHRpbWUK",
 "sync.py": "IiIiClNhdmVzIHRoZSBsZWN0dXJlIG5vdGVzLgoKV3JpdGVzIHRoZSBsaXZlIG5vdGVzIGFzIGEgTWFya2Rvd24gZmlsZSBpbnRvIFNZTkNfRk9MREVSLiBQb2ludCB0aGF0IGZvbGRlciBhdCBHb29nbGUgRHJpdmUsCkRyb3Bib3gsIE9uZURyaXZlIG9yIGlDbG91ZCBEcml2ZSBhbmQgdGhlIG5vdGVzLCBzbGlkZSBwaWN0dXJlcyBhbmQgc3R1ZHkgc2hlZXQgYXBwZWFyIG9uIHlvdXIgcGhvbmUuCiIiIgppbXBvcnQgdGhyZWFkaW5nCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGNvbmZpZwoKCmRlZiBmbXRfdGltZShzZWNvbmRzOiBmbG9hdCkgLT4gc3RyOgogICAgc2Vjb25kcyA9IGludChzZWNvbmRzKQogICAgcmV0dXJuIGYie3NlY29uZHMgLy8gMzYwMDowMmR9OntzZWNvbmRzICUgMzYwMCAvLyA2MDowMmR9OntzZWNvbmRzICUgNjA6MDJkfSIKCgpkZWYgc2hvcnRfdGltZShzZWNvbmRzOiBmbG9hdCkgLT4gc3RyOgogICAgIiIiMDowNywgMTI6MzEsIDE6MDI6MDkgKGhvdyB0aW1lcyByZWFkIG9uIHRoZSBkYXNoYm9hcmQgYW5kIHBob25lKS4iIiIKICAgIHNlY29uZHMgPSBpbnQoc2Vjb25kcykKICAgIGgsIG0sIHNlYyA9IHNlY29uZHMgLy8gMzYwMCwgc2Vjb25kcyAlIDM2MDAgLy8gNjAsIHNlY29uZHMgJSA2MAogICAgcmV0dXJuIGYie2h9OnttOjAyZH06e3NlYzowMmR9IiBpZiBoIGVsc2UgZiJ7bX06e3NlYzowMmR9IgoKCmNsYXNzIE5vdGVzV3JpdGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvdXJzZTogc3RyLCBzeW5jX2ZvbGRlcjogUGF0aCA9IGNvbmZpZy5TWU5DX0ZPTERFUik6CiAgICAgICAgc2VsZi5zdGFydGVkID0gZGF0ZXRpbWUubm93KCkKICAgICAgICBzdGFtcCA9IHNlbGYuc3RhcnRlZC5zdHJmdGltZSgiJVktJW0tJWRfJUglTSIpCiAgICAgICAgc2FmZV9jb3Vyc2UgPSAiIi5qb2luKGMgaWYgYy5pc2FsbnVtKCkgb3IgYyBpbiAiLV8gIiBlbHNlICJfIiBmb3IgYyBpbiBjb3Vyc2UpLnN0cmlwKCkgb3IgIkxlY3R1cmUiCiAgICAgICAgc2VsZi5mb2xkZXIgPSBQYXRoKHN5bmNfZm9sZGVyKS5leHBhbmR1c2VyKCkgLyBmIntzdGFtcH1fe3NhZmVfY291cnNlLnJlcGxhY2UoJyAnLCAnXycpfSIKICAgICAgICBzZWxmLmZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2VsZi5jb3Vyc2UgPSBjb3Vyc2UKICAgICAgICBzZWxmLm5vdGVzX3BhdGggPSBzZWxmLmZvbGRlciAvICJsaXZlX25vdGVzLm1kIgogICAgICAgIHNlbGYuZW50cmllcyA9IFtdICAgICAgICAgICMgKHNlY29uZHMsIGtpbmQsIHRleHQpICBraW5kID0gInNwZWVjaCIgfCAic2xpZGUiIHwgInFhIgogICAgICAgIHNlbGYuX2xvY2sgPSB0aHJlYWRpbmcuTG9jaygpCiAgICAgICAgc2VsZi52ZXJzaW9uID0gMCAgICAgICAgICAgIyBnb2VzIHVwIG9uIGV2ZXJ5IGNoYW5nZSwgc28gdGhlIGRhc2hib2FyZCBrbm93cyB3aGVuIHRvIHJlZnJlc2gKICAgICAgICBzZWxmLl93cml0ZSgpCgogICAgIyAtLS0tLSBhZGRpbmcgY29udGVudCAtLS0tLQogICAgZGVmIGFkZF9zcGVlY2goc2VsZiwgdDogZmxvYXQsIHRleHQ6IHN0cik6CiAgICAgICAgc2VsZi5fYWRkKHQsICJzcGVlY2giLCB0ZXh0KQoKICAgIGRlZiBhZGRfc2xpZGUoc2VsZiwgdDogZmxvYXQsIGltYWdlX3BhdGg6IHN0ciwgdGV4dDogc3RyKToKICAgICAgICByZWwgPSBQYXRoKGltYWdlX3BhdGgpLm5hbWUKICAgICAgICBzZWxmLl9hZGQodCwgInNsaWRlIiwgZiIhW3NsaWRlXSh7cmVsfSlcblxuIiArIChmIj4gIiArIHRleHQucmVwbGFjZSgiXG4iLCAiXG4+ICIpIGlmIHRleHQgZWxzZSAiIikpCgogICAgZGVmIGFkZF9xYShzZWxmLCB0OiBmbG9hdCwgcXVlc3Rpb246IHN0ciwgYW5zd2VyOiBzdHIpOgogICAgICAgIHNlbGYuX2FkZCh0LCAicWEiLCBmIioqWW91IGFza2VkOioqIHtxdWVzdGlvbn1cblxuKipBc3Npc3RhbnQ6Kioge2Fuc3dlcn0iKQoKICAgIGRlZiBfYWRkKHNlbGYsIHQsIGtpbmQsIHRleHQpOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgc2VsZi5lbnRyaWVzLmFwcGVuZCgodCwga2luZCwgdGV4dCkpCiAgICAgICAgICAgIHNlbGYuZW50cmllcy5zb3J0KGtleT1sYW1iZGEgZTogZVswXSkgICAjIHRyYW5zY3JpcHRpb24gY2FuIGxhZywgc28ga2VlcCB0aW1lIG9yZGVyCiAgICAgICAgICAgIHNlbGYudmVyc2lvbiArPSAxCiAgICAgICAgICAgIHNlbGYuX3dyaXRlKCkKCiAgICBkZWYgc3RydWN0dXJlZChzZWxmKToKICAgICAgICAiIiJFbnRyaWVzIGFzIHNpbXBsZSBkaWN0cyBmb3IgdGhlIGRhc2hib2FyZC4iIiIKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIGVudHJpZXMgPSBsaXN0KHNlbGYuZW50cmllcykKICAgICAgICBvdXQgPSBbXQogICAgICAgIGZvciB0LCBraW5kLCB0ZXh0IGluIGVudHJpZXM6CiAgICAgICAgICAgIGlmIGtpbmQgPT0gInNwZWVjaCI6CiAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKHsidCI6IHQsICJraW5kIjoga2luZCwgInRleHQiOiB0ZXh0fSkKICAgICAgICAgICAgZWxpZiBraW5kID09ICJzbGlkZSI6CiAgICAgICAgICAgICAgICBpbWcgPSB0ZXh0LnNwbGl0KCJdKCIsIDEpWzFdLnNwbGl0KCIpIiwgMSlbMF0gaWYgIl0oIiBpbiB0ZXh0IGVsc2UgIiIKICAgICAgICAgICAgICAgIHdvcmRzID0gdGV4dC5zcGxpdCgiXG5cbiIsIDEpWzFdLnJlcGxhY2UoIj4gIiwgIiIpIGlmICJcblxuIiBpbiB0ZXh0IGVsc2UgIiIKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoeyJ0IjogdCwgImtpbmQiOiBraW5kLCAiaW1nIjogaW1nLCAidGV4dCI6IHdvcmRzfSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHEsIF8sIGEgPSB0ZXh0LnBhcnRpdGlvbigiXG5cbiIpCiAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKHsidCI6IHQsICJraW5kIjoga2luZCwgInEiOiBxLnJlcGxhY2UoIioqWW91IGFza2VkOioqICIsICIiKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJhIjogYS5yZXBsYWNlKCIqKkFzc2lzdGFudDoqKiAiLCAiIil9KQogICAgICAgIHJldHVybiBvdXQKCiAgICAjIC0tLS0tIHJlYWRpbmcgY29udGVudCBiYWNrIC0tLS0tCiAgICBkZWYgdHJhbnNjcmlwdChzZWxmLCBsYXN0X3NlY29uZHM6IGZsb2F0ID0gTm9uZSkgLT4gc3RyOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgZW50cmllcyA9IGxpc3Qoc2VsZi5lbnRyaWVzKQogICAgICAgIGlmIGxhc3Rfc2Vjb25kcyBpcyBub3QgTm9uZSBhbmQgZW50cmllczoKICAgICAgICAgICAgY3V0b2ZmID0gZW50cmllc1stMV1bMF0gLSBsYXN0X3NlY29uZHMKICAgICAgICAgICAgZW50cmllcyA9IFtlIGZvciBlIGluIGVudHJpZXMgaWYgZVswXSA+PSBjdXRvZmZdCiAgICAgICAgcmV0dXJuICIgIi5qb2luKHRleHQgZm9yIF8sIGtpbmQsIHRleHQgaW4gZW50cmllcyBpZiBraW5kID09ICJzcGVlY2giKQoKICAgIGRlZiBzbGlkZV90ZXh0KHNlbGYsIGxhc3Rfb25seTogYm9vbCA9IEZhbHNlKSAtPiBzdHI6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBzbGlkZXMgPSBbdGV4dCBmb3IgXywga2luZCwgdGV4dCBpbiBzZWxmLmVudHJpZXMgaWYga2luZCA9PSAic2xpZGUiXQogICAgICAgIGlmIGxhc3Rfb25seToKICAgICAgICAgICAgc2xpZGVzID0gc2xpZGVzWy0xOl0KICAgICAgICByZXR1cm4gIlxuIi5qb2luKHMuc3BsaXQoIlxuXG4iLCAxKVstMV0ucmVwbGFjZSgiPiAiLCAiIikgZm9yIHMgaW4gc2xpZGVzKQoKICAgIGRlZiB0aW1lZF90cmFuc2NyaXB0KHNlbGYpIC0+IHN0cjoKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHJldHVybiAiXG4iLmpvaW4oZiJbe2ZtdF90aW1lKHQpfV0ge3RleHR9IiBmb3IgdCwga2luZCwgdGV4dCBpbiBzZWxmLmVudHJpZXMgaWYga2luZCA9PSAic3BlZWNoIikKCiAgICAjIC0tLS0tIGZpbGUgb3V0cHV0IC0tLS0tCiAgICBkZWYgX3JlbmRlcl9tZChzZWxmKSAtPiBzdHI6CiAgICAgICAgb3V0ID0gW2YiIyB7c2VsZi5jb3Vyc2V9IC0gbGl2ZSBub3RlcyIsCiAgICAgICAgICAgICAgIGYiKlN0YXJ0ZWQge3NlbGYuc3RhcnRlZC5zdHJmdGltZSgnJUEgJUIgJWQsICVZIGF0ICVJOiVNICVwJyl9KiIsICIiXQogICAgICAgIGZvciB0LCBraW5kLCB0ZXh0IGluIHNlbGYuZW50cmllczoKICAgICAgICAgICAgaWYga2luZCA9PSAic3BlZWNoIjoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoZiJge2ZtdF90aW1lKHQpfWAge3RleHR9ICAiKQogICAgICAgICAgICBlbGlmIGtpbmQgPT0gInNsaWRlIjoKICAgICAgICAgICAgICAgIG91dCArPSBbIiIsIGYiIyMjIFNsaWRlIGNhcHR1cmVkIGF0IHtmbXRfdGltZSh0KX0iLCB0ZXh0LCAiIl0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIG91dCArPSBbIiIsIGYiIyMjIyBRdWVzdGlvbiBhdCB7Zm10X3RpbWUodCl9IiwgdGV4dCwgIiJdCiAgICAgICAgcmV0dXJuICJcbiIuam9pbihvdXQpICsgIlxuIgoKICAgIGRlZiBfd3JpdGUoc2VsZik6CiAgICAgICAgdG1wID0gc2VsZi5ub3Rlc19wYXRoLndpdGhfc3VmZml4KCIudG1wIikKICAgICAgICB0bXAud3JpdGVfdGV4dChzZWxmLl9yZW5kZXJfbWQoKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICB0bXAucmVwbGFjZShzZWxmLm5vdGVzX3BhdGgpICAgIyBhdG9taWMsIHNvIGNsb3VkIHN5bmMgbmV2ZXIgZ3JhYnMgYSBoYWxmLXdyaXR0ZW4gZmlsZQo=",
 "vision.py": "IiIiCkNhbWVyYSAtPiBzbGlkZSAvIHdoaXRlYm9hcmQgc25hcHNob3RzICsgT0NSIHRleHQgICh0aGUgVGhpcmQgRXllIGNhbWVyYSkuCgpJZGVhOiBjb21wYXJlIGVhY2ggZnJhbWUgdG8gdGhlIGxhc3Qgc2F2ZWQgc2xpZGUuIFdoZW4gdGhlIHZpZXcgY2hhbmdlcyBhIGxvdCAobmV3IHNsaWRlLApuZXcgd2hpdGVib2FyZCB3cml0aW5nKSBBTkQgdGhlbiBob2xkcyBzdGlsbCBmb3IgYSBtb21lbnQsIHNhdmUgYSBzbmFwc2hvdCBhbmQgcmVhZCBpdHMgdGV4dC4KIiIiCmltcG9ydCBxdWV1ZQppbXBvcnQgdGhyZWFkaW5nCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKCmltcG9ydCBjb25maWcKCnRyeToKICAgIGltcG9ydCBweXRlc3NlcmFjdAogICAgcHl0ZXNzZXJhY3QuZ2V0X3Rlc3NlcmFjdF92ZXJzaW9uKCkgICAjIGNoZWNrcyB0aGUgVGVzc2VyYWN0IHByb2dyYW0gaXMgaW5zdGFsbGVkIHRvbwogICAgX09DUiA9ICJ0ZXNzZXJhY3QiCmV4Y2VwdCBFeGNlcHRpb246CiAgICB0cnk6CiAgICAgICAgaW1wb3J0IGVhc3lvY3IgICMgdHlwZTogaWdub3JlCiAgICAgICAgX3JlYWRlciA9IGVhc3lvY3IuUmVhZGVyKFsiZW4iXSwgZ3B1PUZhbHNlLCB2ZXJib3NlPUZhbHNlKQogICAgICAgIF9PQ1IgPSAiZWFzeW9jciIKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgX09DUiA9IE5vbmUKCgpkZWYgb2NyX2ltYWdlKGltZykgLT4gc3RyOgogICAgaWYgX09DUiA9PSAidGVzc2VyYWN0IjoKICAgICAgICBncmF5ID0gY3YyLmN2dENvbG9yKGltZywgY3YyLkNPTE9SX0JHUjJHUkFZKQogICAgICAgIGdyYXkgPSBjdjIucmVzaXplKGdyYXksIE5vbmUsIGZ4PTEuNSwgZnk9MS41LCBpbnRlcnBvbGF0aW9uPWN2Mi5JTlRFUl9DVUJJQykKICAgICAgICBncmF5ID0gY3YyLmFkYXB0aXZlVGhyZXNob2xkKGdyYXksIDI1NSwgY3YyLkFEQVBUSVZFX1RIUkVTSF9HQVVTU0lBTl9DLCBjdjIuVEhSRVNIX0JJTkFSWSwgMzEsIDE1KQogICAgICAgIHRleHQgPSBweXRlc3NlcmFjdC5pbWFnZV90b19zdHJpbmcoZ3JheSkKICAgIGVsaWYgX09DUiA9PSAiZWFzeW9jciI6CiAgICAgICAgdGV4dCA9ICJcbiIuam9pbihfcmVhZGVyLnJlYWR0ZXh0KGltZywgZGV0YWlsPTApKQogICAgZWxzZToKICAgICAgICByZXR1cm4gIiIKICAgICMga2VlcCBsaW5lcyB0aGF0IGxvb2sgbGlrZSByZWFsIHdvcmRzLCBkcm9wIE9DUiBub2lzZQogICAgbGluZXMgPSBbbG4uc3RyaXAoKSBmb3IgbG4gaW4gdGV4dC5zcGxpdGxpbmVzKCldCiAgICBsaW5lcyA9IFtsbiBmb3IgbG4gaW4gbGluZXMgaWYgbGVuKGxuKSA+PSAzIGFuZCBzdW0oYy5pc2FscGhhKCkgZm9yIGMgaW4gbG4pID49IDAuNSAqIGxlbihsbildCiAgICByZXR1cm4gIlxuIi5qb2luKGxpbmVzKQoKCmRlZiBjaGFuZ2VfcGVyY2VudChhLCBiKSAtPiBmbG9hdDoKICAgICIiIiUgb2YgcGl4ZWxzIHRoYXQgY2xlYXJseSBjaGFuZ2VkIGJldHdlZW4gdHdvIHNtYWxsIGdyYXkgZnJhbWVzLgogICAgV29ya3MgYmV0dGVyIHRoYW4gYW4gYXZlcmFnZSBkaWZmZXJlbmNlIGZvciBzbGlkZXMsIHdoZXJlIG9ubHkgdGhlIHRleHQgY2hhbmdlcy4iIiIKICAgIHJldHVybiAxMDAuMCAqIGZsb2F0KG5wLm1lYW4oY3YyLmFic2RpZmYoYSwgYikgPiAzMCkpCgoKZGVmIF9zbWFsbF9ncmF5KGZyYW1lKToKICAgIGcgPSBjdjIuY3Z0Q29sb3IoZnJhbWUsIGN2Mi5DT0xPUl9CR1IyR1JBWSkKICAgIGcgPSBjdjIucmVzaXplKGcsICgxNjAsIDkwKSkKICAgIHJldHVybiBjdjIuR2F1c3NpYW5CbHVyKGcsICg1LCA1KSwgMCkKCgpjbGFzcyBTbGlkZVdhdGNoZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgb3V0X2RpcjogUGF0aCk6CiAgICAgICAgc2VsZi5vdXRfZGlyID0gUGF0aChvdXRfZGlyKQogICAgICAgIHNlbGYub3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2VsZi5yZXN1bHRzOiAicXVldWUuUXVldWVbdHVwbGVbZmxvYXQsIHN0ciwgc3RyXV0iID0gcXVldWUuUXVldWUoKSAgIyAodCwgaW1hZ2VfcGF0aCwgdGV4dCkKICAgICAgICBzZWxmLl9sYXN0X3NhdmVkID0gTm9uZQogICAgICAgIHNlbGYuX3ByZXYgPSBOb25lCiAgICAgICAgc2VsZi5fc3RhYmxlID0gMAogICAgICAgIHNlbGYuX2xhc3RfdGltZSA9IC0xZTkKICAgICAgICBzZWxmLmNvdW50ID0gMAogICAgICAgIGlmIF9PQ1IgaXMgTm9uZToKICAgICAgICAgICAgcHJpbnQoIlt2aXNpb25dIG5vIE9DUiBlbmdpbmUgZm91bmQgLSBzbGlkZXMgd2lsbCBiZSBzYXZlZCBhcyBpbWFnZXMgd2l0aG91dCB0ZXh0LiAiCiAgICAgICAgICAgICAgICAgICJJbnN0YWxsIFRlc3NlcmFjdCB0byByZWFkIHNsaWRlIHRleHQuIikKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcmludChmIlt2aXNpb25dIE9DUiByZWFkeSAoe19PQ1J9KSIpCgogICAgZGVmIHByb2Nlc3Moc2VsZiwgZnJhbWUsIHQ6IGZsb2F0KToKICAgICAgICAiIiJDYWxsIG9uY2UgcGVyIGNhbWVyYSBmcmFtZS4gdCA9IHNlY29uZHMgc2luY2UgdGhlIGxlY3R1cmUgc3RhcnRlZC4iIiIKICAgICAgICBnID0gX3NtYWxsX2dyYXkoZnJhbWUpCiAgICAgICAgaWYgc2VsZi5fcHJldiBpcyBOb25lOgogICAgICAgICAgICBzZWxmLl9wcmV2ID0gZwogICAgICAgICAgICBzZWxmLl9sYXN0X3NhdmVkID0gTm9uZQogICAgICAgIG1vdmluZyA9IGNoYW5nZV9wZXJjZW50KGcsIHNlbGYuX3ByZXYpID4gMS4wICAgIyBjYW1lcmEvc2NlbmUgc3RpbGwgbW92aW5nPwogICAgICAgIHNlbGYuX3ByZXYgPSBnCiAgICAgICAgc2VsZi5fc3RhYmxlID0gMCBpZiBtb3ZpbmcgZWxzZSBzZWxmLl9zdGFibGUgKyAxCgogICAgICAgIGNoYW5nZWQgPSBzZWxmLl9sYXN0X3NhdmVkIGlzIE5vbmUgb3IgXAogICAgICAgICAgICBjaGFuZ2VfcGVyY2VudChnLCBzZWxmLl9sYXN0X3NhdmVkKSA+IGNvbmZpZy5TTElERV9DSEFOR0VfVEhSRVNIT0xECiAgICAgICAgaWYgY2hhbmdlZCBhbmQgc2VsZi5fc3RhYmxlID49IGNvbmZpZy5TTElERV9TVEFCTEVfRlJBTUVTIGFuZCBcCiAgICAgICAgICAgICAgICB0aW1lLnRpbWUoKSAtIHNlbGYuX2xhc3RfdGltZSA+PSBjb25maWcuTUlOX1NFQ09ORFNfQkVUV0VFTl9TTElERVM6CiAgICAgICAgICAgIHNlbGYuX2xhc3Rfc2F2ZWQgPSBnCiAgICAgICAgICAgIHNlbGYuX2xhc3RfdGltZSA9IHRpbWUudGltZSgpCiAgICAgICAgICAgIHNlbGYuY291bnQgKz0gMQogICAgICAgICAgICBwYXRoID0gc2VsZi5vdXRfZGlyIC8gZiJzbGlkZV97c2VsZi5jb3VudDowM2R9LmpwZyIKICAgICAgICAgICAgY3YyLmltd3JpdGUoc3RyKHBhdGgpLCBmcmFtZSkKICAgICAgICAgICAgIyBPQ1IgaXMgc2xvdywgc28gZG8gaXQgaW4gdGhlIGJhY2tncm91bmQgc28gdGhlIHZpZGVvIG5ldmVyIGZyZWV6ZXMKICAgICAgICAgICAgdGhyZWFkaW5nLlRocmVhZCh0YXJnZXQ9c2VsZi5fb2NyLCBhcmdzPShmcmFtZS5jb3B5KCksIHQsIHN0cihwYXRoKSksIGRhZW1vbj1UcnVlKS5zdGFydCgpCgogICAgZGVmIF9vY3Ioc2VsZiwgZnJhbWUsIHQsIHBhdGgpOgogICAgICAgIHNlbGYucmVzdWx0cy5wdXQoKHQsIHBhdGgsIG9jcl9pbWFnZShmcmFtZSkpKQo=",
 "assistant.py": "IiIiCkxpdmUgYXNzaXN0YW50OiB5b3UgYXNrIGEgcXVlc3Rpb24gb3V0IGxvdWQsIGl0IGFuc3dlcnMgb3V0IGxvdWQuCgpVc2VzIENsYXVkZSAoQU5USFJPUElDX0FQSV9LRVkpIG9yIEdvb2dsZSBHZW1pbmkgKEdFTUlOSV9BUElfS0VZLCB3aGljaCBoYXMgYSBmcmVlIHRpZXIpLgpXaXRob3V0IGEga2V5IGl0IGZhbGxzIGJhY2sgdG8gc2VhcmNoaW5nIHRoZSBsZWN0dXJlIHRyYW5zY3JpcHQgZm9yIHRoZSBtb3N0IHJlbGV2YW50IHNlbnRlbmNlcywKc28gdGhlIGRlbW8gc3RpbGwgd29ya3Mgb2ZmbGluZS4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgcmUKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgdGhyZWFkaW5nCmltcG9ydCB0aW1lCmltcG9ydCB1cmxsaWIuZXJyb3IKaW1wb3J0IHVybGxpYi5yZXF1ZXN0CgppbXBvcnQgY29uZmlnCgpTVE9QV09SRFMgPSBzZXQoIiIiYSBhbiB0aGUgYW5kIG9yIGJ1dCBpZiBvZiB0byBpbiBvbiBhdCBmb3IgZnJvbSBieSB3aXRoIGFib3V0IGFzIGlzIGFyZSB3YXMgd2VyZSBiZSBiZWVuCmJlaW5nIGl0IGl0cyB0aGlzIHRoYXQgdGhlc2UgdGhvc2Ugd2hhdCB3aGljaCB3aG8gd2hvbSB3aG9zZSB3aGVuIHdoZXJlIHdoeSBob3cgZG8gZG9lcyBkaWQgY2FuIGNvdWxkCndvdWxkIHNob3VsZCB3aWxsIHNoYWxsIG1heSBtaWdodCBtdXN0IGkgeW91IGhlIHNoZSB3ZSB0aGV5IG1lIGhpbSBoZXIgdXMgdGhlbSBteSB5b3VyIG91ciB0aGVpciBqdXN0CnNvIHRoYW4gdGhlbiB0aGVyZSBoZXJlIG5vdCBubyB5ZXMgYWxzbyB2ZXJ5IHJlYWxseSBva2F5IG9rIHVtIHVoIGxpa2UgZ29pbmcgZ29ubmEga25vdyIiIi5zcGxpdCgpKQoKCmRlZiBjYWxsX2NsYXVkZShzeXN0ZW06IHN0ciwgdXNlcjogc3RyLCBtYXhfdG9rZW5zOiBpbnQgPSA2MDApIC0+IHN0cjoKICAgICIiIk1pbmltYWwgQW50aHJvcGljIE1lc3NhZ2VzIEFQSSBjYWxsIHVzaW5nIG9ubHkgdGhlIHN0YW5kYXJkIGxpYnJhcnkuIiIiCiAgICBib2R5ID0ganNvbi5kdW1wcyh7CiAgICAgICAgIm1vZGVsIjogY29uZmlnLkFJX01PREVMLAogICAgICAgICJtYXhfdG9rZW5zIjogbWF4X3Rva2VucywKICAgICAgICAic3lzdGVtIjogc3lzdGVtLAogICAgICAgICJtZXNzYWdlcyI6IFt7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogdXNlcn1dLAogICAgfSkuZW5jb2RlKCkKICAgIHJlcSA9IHVybGxpYi5yZXF1ZXN0LlJlcXVlc3QoCiAgICAgICAgImh0dHBzOi8vYXBpLmFudGhyb3BpYy5jb20vdjEvbWVzc2FnZXMiLCBkYXRhPWJvZHksIG1ldGhvZD0iUE9TVCIsCiAgICAgICAgaGVhZGVycz17IngtYXBpLWtleSI6IGNvbmZpZy5BTlRIUk9QSUNfQVBJX0tFWSwgImFudGhyb3BpYy12ZXJzaW9uIjogIjIwMjMtMDYtMDEiLAogICAgICAgICAgICAgICAgICJjb250ZW50LXR5cGUiOiAiYXBwbGljYXRpb24vanNvbiJ9KQogICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKHJlcSwgdGltZW91dD02MCkgYXMgcmVzcDoKICAgICAgICBkYXRhID0ganNvbi5sb2FkKHJlc3ApCiAgICByZXR1cm4gIiIuam9pbihibG9jay5nZXQoInRleHQiLCAiIikgZm9yIGJsb2NrIGluIGRhdGEuZ2V0KCJjb250ZW50IiwgW10pKS5zdHJpcCgpCgoKIyBMaXZlIHF1ZXN0aW9ucyB1c2UgdGhlIGZhc3QgIkxpdGUiIG1vZGVscyAodGhpbmtpbmcgaXMgb2ZmIGJ5IGRlZmF1bHQsIHNvIGFuc3dlcnMgY29tZSBiYWNrIHF1aWNrbHkpLgojIFN0dWR5IHNoZWV0cyB1c2UgdGhlIHNtYXJ0ZXIgRmxhc2ggbW9kZWxzLiBBbGwgb2YgdGhlc2UgaGF2ZSBhIGZyZWUgdGllci4KR0VNSU5JX0ZBU1QgPSBbImdlbWluaS0yLjUtZmxhc2gtbGl0ZSIsICJnZW1pbmktMy4xLWZsYXNoLWxpdGUiLCAiZ2VtaW5pLTIuNS1mbGFzaCIsICJnZW1pbmktMy41LWZsYXNoIl0KR0VNSU5JX1NNQVJUID0gWyJnZW1pbmktMi41LWZsYXNoIiwgImdlbWluaS0zLjUtZmxhc2giLCAiZ2VtaW5pLTMuMS1mbGFzaC1saXRlIiwgImdlbWluaS0yLjUtZmxhc2gtbGl0ZSJdCl9sYXN0X2dvb2QgPSB7fSAgICAgICAgICAgICMgImZhc3QiLyJzbWFydCIgLT4gbW9kZWwgdGhhdCBhbnN3ZXJlZCBsYXN0IHRpbWUgKHRyaWVkIGZpcnN0IG5leHQgdGltZSkKX25vX3RoaW5raW5nX2NmZyA9IHNldCgpICAgIyBtb2RlbHMgdGhhdCByZWplY3RlZCB0aGUgdGhpbmtpbmcgc2V0dGluZwoKCmRlZiBfZ2VtaW5pX3JlcXVlc3QobW9kZWwsIHN5c3RlbSwgdXNlciwgbWF4X3Rva2VucywgdGltZW91dCk6CiAgICBnZW4gPSB7Im1heE91dHB1dFRva2VucyI6IG1heF90b2tlbnMgKiA0ICsgMTAyNCwgInRlbXBlcmF0dXJlIjogMC40fQogICAgIyBUaGlua2luZyBtYWtlcyBhbnN3ZXJzIHNsb3dlci4gMi41IEZsYXNoLUxpdGUgYWxyZWFkeSBoYXMgaXQgb2ZmOyBhc2sgdGhlIG90aGVycyB0byBrZWVwIGl0IGxvdy4KICAgIGlmIG1vZGVsICE9ICJnZW1pbmktMi41LWZsYXNoLWxpdGUiIGFuZCBtb2RlbCBub3QgaW4gX25vX3RoaW5raW5nX2NmZzoKICAgICAgICBnZW5bInRoaW5raW5nQ29uZmlnIl0gPSB7InRoaW5raW5nTGV2ZWwiOiAibG93In0KICAgIGJvZHkgPSB7InN5c3RlbUluc3RydWN0aW9uIjogeyJwYXJ0cyI6IFt7InRleHQiOiBzeXN0ZW19XX0sCiAgICAgICAgICAgICJjb250ZW50cyI6IFt7InJvbGUiOiAidXNlciIsICJwYXJ0cyI6IFt7InRleHQiOiB1c2VyfV19XSwKICAgICAgICAgICAgImdlbmVyYXRpb25Db25maWciOiBnZW59CiAgICByZXEgPSB1cmxsaWIucmVxdWVzdC5SZXF1ZXN0KAogICAgICAgIGYiaHR0cHM6Ly9nZW5lcmF0aXZlbGFuZ3VhZ2UuZ29vZ2xlYXBpcy5jb20vdjFiZXRhL21vZGVscy97bW9kZWx9OmdlbmVyYXRlQ29udGVudCIsCiAgICAgICAgZGF0YT1qc29uLmR1bXBzKGJvZHkpLmVuY29kZSgpLCBtZXRob2Q9IlBPU1QiLAogICAgICAgIGhlYWRlcnM9eyJ4LWdvb2ctYXBpLWtleSI6IGNvbmZpZy5HRU1JTklfQVBJX0tFWSwgImNvbnRlbnQtdHlwZSI6ICJhcHBsaWNhdGlvbi9qc29uIn0pCiAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxLCB0aW1lb3V0PXRpbWVvdXQpIGFzIHJlc3A6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZChyZXNwKQoKCmRlZiBjYWxsX2dlbWluaShzeXN0ZW06IHN0ciwgdXNlcjogc3RyLCBtYXhfdG9rZW5zOiBpbnQgPSA2MDAsIGZhc3Q6IGJvb2wgPSBGYWxzZSkgLT4gc3RyOgogICAgIiIiR29vZ2xlIEdlbWluaSBBUEkgKGZyZWUgdGllciBhdmFpbGFibGUpLCBzdGFuZGFyZCBsaWJyYXJ5IG9ubHkuIiIiCiAgICBtb2RlID0gImZhc3QiIGlmIGZhc3QgZWxzZSAic21hcnQiCiAgICBtb2RlbHMgPSBbY29uZmlnLkdFTUlOSV9NT0RFTF0gaWYgY29uZmlnLkdFTUlOSV9NT0RFTCBhbmQgbm90IGZhc3QgZWxzZSBbXQogICAgbW9kZWxzID0gbGlzdChkaWN0LmZyb21rZXlzKChbX2xhc3RfZ29vZFttb2RlXV0gaWYgbW9kZSBpbiBfbGFzdF9nb29kIGVsc2UgW10pICsgbW9kZWxzCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKyAoR0VNSU5JX0ZBU1QgaWYgZmFzdCBlbHNlIEdFTUlOSV9TTUFSVCkpKQogICAgdGltZW91dCA9IDIwIGlmIGZhc3QgZWxzZSA5MAogICAgbGFzdF9lcnJvciA9IE5vbmUKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDIpOiAgICAgICAgICAgICAgICAgICMgdHdvIHJvdW5kcyB0aHJvdWdoIHRoZSBtb2RlbHMsIHdpdGggYSBzaG9ydCBwYXVzZSBiZXR3ZWVuCiAgICAgICAgZm9yIG1vZGVsIGluIG1vZGVsczoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIGRhdGEgPSBfZ2VtaW5pX3JlcXVlc3QobW9kZWwsIHN5c3RlbSwgdXNlciwgbWF4X3Rva2VucywgdGltZW91dCkKICAgICAgICAgICAgICAgIGV4Y2VwdCB1cmxsaWIuZXJyb3IuSFRUUEVycm9yIGFzIGU6CiAgICAgICAgICAgICAgICAgICAgaWYgZS5jb2RlID09IDQwMCBhbmQgbW9kZWwgbm90IGluIF9ub190aGlua2luZ19jZmc6CiAgICAgICAgICAgICAgICAgICAgICAgIF9ub190aGlua2luZ19jZmcuYWRkKG1vZGVsKSAgICMgbW9kZWwgZGlkbid0IGFjY2VwdCB0aGUgdGhpbmtpbmcgc2V0dGluZzogcmV0cnkgd2l0aG91dAogICAgICAgICAgICAgICAgICAgICAgICBkYXRhID0gX2dlbWluaV9yZXF1ZXN0KG1vZGVsLCBzeXN0ZW0sIHVzZXIsIG1heF90b2tlbnMsIHRpbWVvdXQpCiAgICAgICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgZXhjZXB0IHVybGxpYi5lcnJvci5IVFRQRXJyb3IgYXMgZToKICAgICAgICAgICAgICAgIGRldGFpbCA9IGUucmVhZCgpLmRlY29kZShlcnJvcnM9Imlnbm9yZSIpWzozMDBdCiAgICAgICAgICAgICAgICBpZiBlLmNvZGUgaW4gKDQwNCwgNTAwLCA1MDMsIDUwNCk6ICAjIG1vZGVsIG1pc3Npbmcgb3IgYnVzeTogdHJ5IHRoZSBuZXh0IG9uZQogICAgICAgICAgICAgICAgICAgIGxhc3RfZXJyb3IgPSBmInttb2RlbH06IHsnbm90IGF2YWlsYWJsZScgaWYgZS5jb2RlID09IDQwNCBlbHNlICdidXN5IHJpZ2h0IG5vdyd9IgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBpZiBlLmNvZGUgPT0gNDI5OiAgICAgICAgICAgICAgICAgICAgICMgbGltaXQgZm9yIHRoaXMgbW9kZWw6IGFub3RoZXIgbW9kZWwgaGFzIGl0cyBvd24gbGltaXQKICAgICAgICAgICAgICAgICAgICBsYXN0X2Vycm9yID0gIkdlbWluaSBmcmVlLXRpZXIgbGltaXQgcmVhY2hlZDsgd2FpdCBhIG1pbnV0ZSBhbmQgdHJ5IGFnYWluIgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJHZW1pbmkgZXJyb3Ige2UuY29kZX06IHtkZXRhaWx9IikgZnJvbSBOb25lCiAgICAgICAgICAgIGV4Y2VwdCAodXJsbGliLmVycm9yLlVSTEVycm9yLCBUaW1lb3V0RXJyb3IpIGFzIGU6CiAgICAgICAgICAgICAgICBsYXN0X2Vycm9yID0gZiJjb3VsZG4ndCByZWFjaCB7bW9kZWx9ICh7ZX0pIgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcGFydHMgPSBkYXRhLmdldCgiY2FuZGlkYXRlcyIsIFt7fV0pWzBdLmdldCgiY29udGVudCIsIHt9KS5nZXQoInBhcnRzIiwgW10pCiAgICAgICAgICAgIHRleHQgPSAiIi5qb2luKHAuZ2V0KCJ0ZXh0IiwgIiIpIGZvciBwIGluIHBhcnRzIGlmIG5vdCBwLmdldCgidGhvdWdodCIpKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIHRleHQ6CiAgICAgICAgICAgICAgICBfbGFzdF9nb29kW21vZGVdID0gbW9kZWwKICAgICAgICAgICAgICAgIHJldHVybiB0ZXh0CiAgICAgICAgICAgIGxhc3RfZXJyb3IgPSBmInttb2RlbH0gcmV0dXJuZWQgYW4gZW1wdHkgYW5zd2VyIgogICAgICAgIHRpbWUuc2xlZXAoMSkKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihsYXN0X2Vycm9yIG9yICJHZW1pbmkgZGlkbid0IGFuc3dlciIpCgoKZGVmIGNhbGxfYWkoc3lzdGVtOiBzdHIsIHVzZXI6IHN0ciwgbWF4X3Rva2VuczogaW50ID0gNjAwLCBmYXN0OiBib29sID0gRmFsc2UpIC0+IHN0cjoKICAgICIiIlVzZSB3aGljaGV2ZXIgQUkgaGFzIGEga2V5OiBDbGF1ZGUgZmlyc3QsIHRoZW4gR2VtaW5pLiBmYXN0PVRydWUgZm9yIGxpdmUgcXVlc3Rpb25zLiIiIgogICAgaWYgY29uZmlnLkFOVEhST1BJQ19BUElfS0VZOgogICAgICAgIHJldHVybiBjYWxsX2NsYXVkZShzeXN0ZW0sIHVzZXIsIG1heF90b2tlbnMpCiAgICBpZiBjb25maWcuR0VNSU5JX0FQSV9LRVk6CiAgICAgICAgcmV0dXJuIGNhbGxfZ2VtaW5pKHN5c3RlbSwgdXNlciwgbWF4X3Rva2VucywgZmFzdD1mYXN0KQogICAgcmFpc2UgUnVudGltZUVycm9yKCJubyBBSSBrZXkgc2V0IikKCgpkZWYga2V5d29yZHModGV4dDogc3RyKToKICAgIHJldHVybiBbdyBmb3IgdyBpbiByZS5maW5kYWxsKHIiW2Etel1bYS16Jy1dKyIsIHRleHQubG93ZXIoKSkgaWYgdyBub3QgaW4gU1RPUFdPUkRTIGFuZCBsZW4odykgPiAyXQoKCmRlZiBvZmZsaW5lX2Fuc3dlcihxdWVzdGlvbjogc3RyLCB0cmFuc2NyaXB0OiBzdHIpIC0+IHN0cjoKICAgIHEgPSBzZXQoa2V5d29yZHMocXVlc3Rpb24pKQogICAgc2VudGVuY2VzID0gW3Muc3RyaXAoKSBmb3IgcyBpbiByZS5zcGxpdChyIig/PD1bLiE/XSlccysiLCB0cmFuc2NyaXB0KSBpZiBsZW4ocy5zcGxpdCgpKSA+IDNdCiAgICBpZiBub3QgcSBvciBub3Qgc2VudGVuY2VzOgogICAgICAgIHJldHVybiAiSSBkb24ndCBoYXZlIGVub3VnaCBmcm9tIHRoZSBsZWN0dXJlIHlldCB0byBhbnN3ZXIgdGhhdC4iCiAgICBzY29yZWQgPSBzb3J0ZWQoc2VudGVuY2VzLCBrZXk9bGFtYmRhIHM6IGxlbihxICYgc2V0KGtleXdvcmRzKHMpKSksIHJldmVyc2U9VHJ1ZSkKICAgIGJlc3QgPSBbcyBmb3IgcyBpbiBzY29yZWRbOjJdIGlmIHEgJiBzZXQoa2V5d29yZHMocykpXQogICAgaWYgbm90IGJlc3Q6CiAgICAgICAgcmV0dXJuICJUaGUgcHJvZmVzc29yIGhhc24ndCBtZW50aW9uZWQgdGhhdCB5ZXQgaW4gdGhpcyBsZWN0dXJlLiIKICAgIHJldHVybiAiRnJvbSB0aGUgbGVjdHVyZTogIiArICIgIi5qb2luKGJlc3QpCgoKY2xhc3MgQXNzaXN0YW50OgogICAgU1lTVEVNID0gKCJZb3UgYXJlIGEgZGlzY3JlZXQgc3R1ZHkgYXNzaXN0YW50IHNwZWFraW5nIHRocm91Z2ggYW4gZWFycGllY2UgdG8gYSB1bml2ZXJzaXR5IHN0dWRlbnQgIgogICAgICAgICAgICAgICJkdXJpbmcgYSBsZWN0dXJlLiBUaGUgc3R1ZGVudCBtYXkgaGF2ZSBBREhEIG9yIGJlIHRpcmVkLCBzbyBhbnN3ZXIgaW4gMS0zIHNob3J0LCBjbGVhciAiCiAgICAgICAgICAgICAgInNwb2tlbiBzZW50ZW5jZXMuIE5vIGxpc3RzLCBubyBtYXJrZG93bi4gVXNlIHRoZSBsZWN0dXJlIHRyYW5zY3JpcHQgd2hlbiBpdCBpcyByZWxldmFudDsgIgogICAgICAgICAgICAgICJpZiB5b3UgYWRkIGdlbmVyYWwga25vd2xlZGdlLCBrZWVwIGl0IGNvbnNpc3RlbnQgd2l0aCB3aGF0IHRoZSBwcm9mZXNzb3Igc2FpZC4iKQoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLnByb3ZpZGVyID0gY29uZmlnLmFpX3Byb3ZpZGVyKCkKICAgICAgICBzZWxmLm9ubGluZSA9IGJvb2woc2VsZi5wcm92aWRlcikKICAgICAgICBwcmludChmIlthc3Npc3RhbnRdIHtzZWxmLnByb3ZpZGVyICsgJyBBUEknIGlmIHNlbGYub25saW5lIGVsc2UgJ29mZmxpbmUgbW9kZSAobm8gQUkga2V5IHNldCknfSIpCgogICAgZGVmIGFuc3dlcihzZWxmLCBxdWVzdGlvbjogc3RyLCByZWNlbnRfdHJhbnNjcmlwdDogc3RyLCBzbGlkZV90ZXh0OiBzdHIgPSAiIikgLT4gc3RyOgogICAgICAgIGlmIHNlbGYub25saW5lOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBwcm9tcHQgPSAoZiJSZWNlbnQgbGVjdHVyZSB0cmFuc2NyaXB0Olxue3JlY2VudF90cmFuc2NyaXB0IG9yICcobm90aGluZyB5ZXQpJ31cblxuIgogICAgICAgICAgICAgICAgICAgICAgICAgIGYiVGV4dCBvbiB0aGUgY3VycmVudCBzbGlkZS9ib2FyZDpcbntzbGlkZV90ZXh0IG9yICcobm9uZSknfVxuXG4iCiAgICAgICAgICAgICAgICAgICAgICAgICAgZiJTdHVkZW50J3MgcXVlc3Rpb246IHtxdWVzdGlvbn0iKQogICAgICAgICAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICAgICAgYW5zd2VyID0gY2FsbF9haShzZWxmLlNZU1RFTSwgcHJvbXB0LCBtYXhfdG9rZW5zPTIwMCwgZmFzdD1UcnVlKQogICAgICAgICAgICAgICAgcHJpbnQoZiJbYXNzaXN0YW50XSBhbnN3ZXJlZCBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyIpCiAgICAgICAgICAgICAgICByZXR1cm4gYW5zd2VyCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgIHByaW50KGYiW2Fzc2lzdGFudF0gQVBJIGVycm9yLCB1c2luZyBvZmZsaW5lIGFuc3dlcjoge2V9IikKICAgICAgICByZXR1cm4gb2ZmbGluZV9hbnN3ZXIocXVlc3Rpb24sIHJlY2VudF90cmFuc2NyaXB0KQoKCiMgLS0tLS0tLS0tLSBUZXh0IHRvIHNwZWVjaCAodGhlIGVhcnBpZWNlICJ2b2ljZSIpIC0tLS0tLS0tLS0KX3NwZWFrX2xvY2sgPSB0aHJlYWRpbmcuTG9jaygpCgoKZGVmIHNwZWFrKHRleHQ6IHN0cik6CiAgICAiIiJTYXkgdGV4dCBvdXQgbG91ZCB3aXRob3V0IGZyZWV6aW5nIHRoZSBhcHAuIFVzZXMgdGhlIE9TIHZvaWNlOyBweXR0c3gzIGlmIGluc3RhbGxlZC4iIiIKICAgIGRlZiBfcnVuKCk6CiAgICAgICAgd2l0aCBfc3BlYWtfbG9jazoKICAgICAgICAgICAgc3lzdGVtID0gcGxhdGZvcm0uc3lzdGVtKCkKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgaWYgc3lzdGVtID09ICJEYXJ3aW4iOgogICAgICAgICAgICAgICAgICAgIHN1YnByb2Nlc3MucnVuKFsic2F5IiwgdGV4dF0sIGNoZWNrPUZhbHNlKQogICAgICAgICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICAgICAgaWYgc3lzdGVtID09ICJMaW51eCIgYW5kIHNodXRpbC53aGljaCgiZXNwZWFrIik6CiAgICAgICAgICAgICAgICAgICAgc3VicHJvY2Vzcy5ydW4oWyJlc3BlYWsiLCB0ZXh0XSwgY2hlY2s9RmFsc2UpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgaW1wb3J0IHB5dHRzeDMKICAgICAgICAgICAgICAgICAgICBlbmdpbmUgPSBweXR0c3gzLmluaXQoKQogICAgICAgICAgICAgICAgICAgIGVuZ2luZS5zYXkodGV4dCkKICAgICAgICAgICAgICAgICAgICBlbmdpbmUucnVuQW5kV2FpdCgpCiAgICAgICAgICAgICAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgICAgICAgICAgICAgaWYgc3lzdGVtICE9ICJXaW5kb3dzIjoKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgICAgICAgICAjIFdpbmRvd3MgYnVpbHQtaW4gdm9pY2UsIG5vIGluc3RhbGwgbmVlZGVkCiAgICAgICAgICAgICAgICAgICAgc2FmZSA9IHRleHQucmVwbGFjZSgiJyIsICInJyIpCiAgICAgICAgICAgICAgICAgICAgc3VicHJvY2Vzcy5ydW4oWyJwb3dlcnNoZWxsIiwgIi1Ob1Byb2ZpbGUiLCAiLUNvbW1hbmQiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiQWRkLVR5cGUgLUFzc2VtYmx5TmFtZSBTeXN0ZW0uU3BlZWNoOyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYiKE5ldy1PYmplY3QgU3lzdGVtLlNwZWVjaC5TeW50aGVzaXMuU3BlZWNoU3ludGhlc2l6ZXIpLlNwZWFrKCd7c2FmZX0nKSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNoZWNrPUZhbHNlKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICBwcmludChmIltzcGVha10gKG5vIHZvaWNlIGF2YWlsYWJsZToge2V9KSAtPiB7dGV4dH0iKQogICAgdGhyZWFkaW5nLlRocmVhZCh0YXJnZXQ9X3J1biwgZGFlbW9uPVRydWUpLnN0YXJ0KCkK",
 "study_sheet.py": "IiIiCkFmdGVyIHRoZSBsZWN0dXJlOiBjb25jaXNlIHN0dWR5IHNoZWV0ICsgcmVtaW5kZXJzLgoKT3V0cHV0cyAoaW4gdGhlIGxlY3R1cmUncyBzeW5jZWQgZm9sZGVyKToKICBzdHVkeV9zaGVldC5tZCAgICAtIHJlYWRhYmxlIGFueXdoZXJlCiAgc3R1ZHlfc2hlZXQuZG9jeCAgLSBXb3JkIHZlcnNpb24gKGlmIHB5dGhvbi1kb2N4IGlzIGluc3RhbGxlZCkKICByZW1pbmRlcnMuaWNzICAgICAtIHRhcCBvbiB5b3VyIHBob25lIHRvIGFkZCByZXZpZXcgcmVtaW5kZXJzICsgdG8tZG9zIHRvIHlvdXIgY2FsZW5kYXIKIiIiCmltcG9ydCBqc29uCmltcG9ydCByZQppbXBvcnQgdXVpZApmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lZGVsdGEKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgY29uZmlnCmZyb20gYXNzaXN0YW50IGltcG9ydCBTVE9QV09SRFMsIGNhbGxfYWksIGtleXdvcmRzCgpTSEVFVF9QUk9NUFQgPSAiIiJZb3UgYXJlIG1ha2luZyBhIG9uZS1wYWdlIHN0dWR5IHNoZWV0IGZvciBhIHVuaXZlcnNpdHkgc3R1ZGVudCB3aXRoIEFESEQvaW5zb21uaWEKd2hvIG1heSBoYXZlIHpvbmVkIG91dCBkdXJpbmcgcGFydHMgb2YgdGhpcyBsZWN0dXJlLiBCZSBjb25jaXNlLCBjb25jcmV0ZSBhbmQgc2tpbW1hYmxlLgoKQ291cnNlOiB7Y291cnNlfQpMZWN0dXJlIGRhdGU6IHtkYXRlfQoKVFJBTlNDUklQVCAoYXV0byBzcGVlY2gtdG8tdGV4dCwgbWF5IGNvbnRhaW4gZXJyb3JzKToKe3RyYW5zY3JpcHR9CgpURVhUIENBUFRVUkVEIEZST00gU0xJREVTL1dISVRFQk9BUkQgKE9DUiwgbWF5IGNvbnRhaW4gZXJyb3JzKToKe3NsaWRlc30KClJldHVybiBPTkxZIHZhbGlkIEpTT04gd2l0aCBleGFjdGx5IHRoZXNlIGtleXM6Cnt7CiAidGl0bGUiOiAic2hvcnQgbGVjdHVyZSB0b3BpYyIsCiAidGxkciI6ICIyLTMgc2VudGVuY2Ugc3VtbWFyeSBvZiB0aGUgd2hvbGUgbGVjdHVyZSIsCiAia2V5X3BvaW50cyI6IFsiNS04IG1vc3QgaW1wb3J0YW50IGlkZWFzLCBvbmUgc2hvcnQgc2VudGVuY2UgZWFjaCJdLAogImtleV90ZXJtcyI6IFt7eyJ0ZXJtIjogIi4uLiIsICJkZWZpbml0aW9uIjogInNob3J0IGRlZmluaXRpb24gYXMgdXNlZCBpbiB0aGUgbGVjdHVyZSJ9fV0sCiAiZXhhbV9xdWVzdGlvbnMiOiBbIjMtNSBsaWtlbHkgZXhhbSAvIHNlbGYtdGVzdCBxdWVzdGlvbnMiXSwKICJ0b2RvcyI6IFt7eyJ0YXNrIjogImhvbWV3b3JrLCByZWFkaW5ncywgZGVhZGxpbmVzIHRoZSBwcm9mZXNzb3IgbWVudGlvbmVkIiwgImR1ZV9pbl9kYXlzIjogN319XSwKICJjb25mdXNpbmdfcGFydHMiOiBbImFueXRoaW5nIHRoYXQgc291bmRlZCB1bmNsZWFyIG9yIHdvcnRoIGFza2luZyB0aGUgcHJvZmVzc29yIGFib3V0IChjYW4gYmUgZW1wdHkpIl0KfX0KRm9yIHRvZG9zLCBlc3RpbWF0ZSBkdWVfaW5fZGF5cyBmcm9tIHdoYXQgd2FzIHNhaWQgKGUuZy4gJ25leHQgY2xhc3MnIH4yLCAnbmV4dCB3ZWVrJyA3KS4gVXNlIFtdIGlmIG5vbmUuIiIiCgoKIyAtLS0tLS0tLS0tLS0tLS0tIEFJIHZlcnNpb24gLS0tLS0tLS0tLS0tLS0tLQpkZWYgYWlfc2hlZXQoY291cnNlLCB0cmFuc2NyaXB0LCBzbGlkZXMpOgogICAgcmF3ID0gY2FsbF9haSgiWW91IHdyaXRlIGV4Y2VsbGVudCwgYWNjdXJhdGUgc3R1ZHkgbWF0ZXJpYWxzLiBPdXRwdXQgSlNPTiBvbmx5LiIsCiAgICAgICAgICAgICAgICAgICAgICBTSEVFVF9QUk9NUFQuZm9ybWF0KGNvdXJzZT1jb3Vyc2UsIGRhdGU9ZGF0ZXRpbWUubm93KCkuc3RyZnRpbWUoIiVZLSVtLSVkIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRyYW5zY3JpcHQ9dHJhbnNjcmlwdFstNjAwMDA6XSBvciAiKGVtcHR5KSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNsaWRlcz1zbGlkZXNbLTE1MDAwOl0gb3IgIihub25lKSIpLAogICAgICAgICAgICAgICAgICAgICAgbWF4X3Rva2Vucz0yMDAwKQogICAgbWF0Y2ggPSByZS5zZWFyY2gociJcey4qXH0iLCByYXcsIHJlLlMpCiAgICByZXR1cm4ganNvbi5sb2FkcyhtYXRjaC5ncm91cCgwKSBpZiBtYXRjaCBlbHNlIHJhdykKCgojIC0tLS0tLS0tLS0tLS0tLS0gT2ZmbGluZSB2ZXJzaW9uIChubyBBUEkga2V5KSAtLS0tLS0tLS0tLS0tLS0tCkZJTExFUiA9IHNldCgiIiJvbmUgdHdvIHRocmVlIGZvdXIgZml2ZSBzaXggc2V2ZW4gZWlnaHQgbmluZSB0ZW4gZmlyc3Qgc2Vjb25kIHRoaXJkIG5leHQgbGFzdAp0b2RheSB0YWxrIHRhbGtpbmcgZ29pbmcgY2FsbGVkIGhhcHBlbnMgbWFrZSBtYWtlcyBtYWRlIHRoaW5nIHRoaW5ncyBzb21ldGhpbmcgc3R1ZmYgcmlnaHQgYmFzaWNhbGx5CmFjdHVhbGx5IGtpbmQgc29ydCBsb3QgcGVvcGxlIGludG8gb250byBvdmVyIHVuZGVyIHdhbnQgbmVlZCBsb29rIHNlZSBzYWlkIHNheXMgc2F5IGdvb2QgZ3JlYXQgZXhhbXBsZSBtZWFuIG1lYW5zIGxldCIiIi5zcGxpdCgpKQpXRUVLREFZUyA9IFsibW9uZGF5IiwgInR1ZXNkYXkiLCAid2VkbmVzZGF5IiwgInRodXJzZGF5IiwgImZyaWRheSIsICJzYXR1cmRheSIsICJzdW5kYXkiXQpUT0RPX1dPUkRTID0gcmUuY29tcGlsZShyIlxiKGR1ZXxob21ld29ya3xhc3NpZ25tZW50fHByb2JsZW0gc2V0fHF1aXp8ZXhhbXxtaWR0ZXJtfHJlYWQoaW5nKT98IgogICAgICAgICAgICAgICAgICAgICAgICByImNoYXB0ZXJ8c3VibWl0fGRlYWRsaW5lfGxhYiByZXBvcnR8ZXNzYXl8cHJvamVjdHxmaW5hbCBleGFtKVxiIiwgcmUuSSkKCgpkZWYgX2d1ZXNzX2R1ZV9kYXlzKHNlbnRlbmNlOiBzdHIpIC0+IGludDoKICAgIHMgPSBzZW50ZW5jZS5sb3dlcigpCiAgICBpZiAidG9tb3Jyb3ciIGluIHM6CiAgICAgICAgcmV0dXJuIDEKICAgIGlmICJuZXh0IGNsYXNzIiBpbiBzIG9yICJuZXh0IGxlY3R1cmUiIGluIHM6CiAgICAgICAgcmV0dXJuIDIKICAgIGlmICJuZXh0IHdlZWsiIGluIHM6CiAgICAgICAgcmV0dXJuIDcKICAgIGlmICJ0d28gd2Vla3MiIGluIHM6CiAgICAgICAgcmV0dXJuIDE0CiAgICBmb3IgaSwgZGF5IGluIGVudW1lcmF0ZShXRUVLREFZUyk6CiAgICAgICAgaWYgZGF5IGluIHM6CiAgICAgICAgICAgIHJldHVybiAoaSAtIGRhdGV0aW1lLm5vdygpLndlZWtkYXkoKSkgJSA3IG9yIDcKICAgIHJldHVybiAzCgoKZGVmIG9mZmxpbmVfc2hlZXQoY291cnNlLCB0cmFuc2NyaXB0LCBzbGlkZXMpOgogICAgdGV4dCA9IHRyYW5zY3JpcHQgKyAiICIgKyBzbGlkZXMKICAgIHNlbnRlbmNlcyA9IGxpc3QoZGljdC5mcm9ta2V5cyhzLnN0cmlwKCkgZm9yIHMgaW4gcmUuc3BsaXQociIoPzw9Wy4hP10pXHMrIiwgdHJhbnNjcmlwdCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBsZW4ocy5zcGxpdCgpKSA+PSA2KSkgICAjIG5vIHJlcGVhdHMKICAgIHdvcmRzID0ga2V5d29yZHModGV4dCkKICAgIGZyZXEgPSBDb3VudGVyKHdvcmRzKQogICAgIyBjYW5kaWRhdGUgdGVybXM6IG5vIG51bWJlciB3b3JkcyAvIGZpbGxlciwgbm8gY29udHJhY3Rpb25zCiAgICBkZWYgZ29vZCh3KToKICAgICAgICByZXR1cm4gIiciIG5vdCBpbiB3IGFuZCB3IG5vdCBpbiBGSUxMRVIgYW5kIGxlbih3KSA+IDMKICAgICMgdHdvLXdvcmQgcGhyYXNlcyBhcmUgdXN1YWxseSBiZXR0ZXIgInRlcm1zIiB0aGFuIHNpbmdsZSB3b3JkcwogICAgYmlncmFtcyA9IENvdW50ZXIoZiJ7YX0ge2J9IiBmb3IgYSwgYiBpbiB6aXAod29yZHMsIHdvcmRzWzE6XSkgaWYgYSAhPSBiIGFuZCBnb29kKGEpIGFuZCBnb29kKGIpKQogICAgdGVybXMsIHVzZWQgPSBbXSwgc2V0KCkKICAgIGZvciBiLCBjIGluIGJpZ3JhbXMubW9zdF9jb21tb24oMTUpOgogICAgICAgIGlmIGMgPj0gMiBhbmQgbm90IChzZXQoYi5zcGxpdCgpKSAmIHVzZWQpIGFuZCBsZW4odGVybXMpIDwgNDoKICAgICAgICAgICAgdGVybXMuYXBwZW5kKGIpCiAgICAgICAgICAgIHVzZWQgfD0gc2V0KGIuc3BsaXQoKSkKICAgIGZvciB3LCBjIGluIGZyZXEubW9zdF9jb21tb24oNDApOgogICAgICAgIGlmIGxlbih0ZXJtcykgPj0gOCBvciBjIDwgMjoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBnb29kKHcpIGFuZCB3IG5vdCBpbiB1c2VkOgogICAgICAgICAgICB0ZXJtcy5hcHBlbmQodykKICAgICAgICAgICAgdXNlZC5hZGQodykKCiAgICBkZWYgc2NvcmUocyk6CiAgICAgICAga3dzID0ga2V5d29yZHMocykKICAgICAgICByZXR1cm4gc3VtKGZyZXFbd10gZm9yIHcgaW4ga3dzKSAvIChsZW4oa3dzKSArIDMpCgogICAgdG9wID0gc29ydGVkKHNlbnRlbmNlcywga2V5PXNjb3JlLCByZXZlcnNlPVRydWUpWzo3XQogICAga2V5X3BvaW50cyA9IFtzIGZvciBzIGluIHNlbnRlbmNlcyBpZiBzIGluIHRvcF0gICAgICAgICMga2VlcCBsZWN0dXJlIG9yZGVyCgogICAgZGVmIGRlZmluZSh0ZXJtKToKICAgICAgICBmb3IgcyBpbiBzZW50ZW5jZXM6CiAgICAgICAgICAgIGlmIHRlcm0gaW4gcy5sb3dlcigpOgogICAgICAgICAgICAgICAgcmV0dXJuIHMKICAgICAgICByZXR1cm4gIihtZW50aW9uZWQgb24gc2xpZGVzKSIKCiAgICB0b2RvX3NlbnRzID0gZGljdC5mcm9ta2V5cyhzLnN0cmlwKCkgZm9yIHMgaW4gcmUuc3BsaXQociIoPzw9Wy4hP10pXHMrIiwgdHJhbnNjcmlwdCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIFRPRE9fV09SRFMuc2VhcmNoKHMpIGFuZCBsZW4ocy5zcGxpdCgpKSA+PSA0KQogICAgdG9kb3MgPSBbeyJ0YXNrIjogcywgImR1ZV9pbl9kYXlzIjogX2d1ZXNzX2R1ZV9kYXlzKHMpfSBmb3IgcyBpbiB0b2RvX3NlbnRzXQogICAgcmV0dXJuIHsKICAgICAgICAidGl0bGUiOiBjb3Vyc2UsCiAgICAgICAgInRsZHIiOiAiICIuam9pbihrZXlfcG9pbnRzWzoyXSkgb3IgIk5vIHNwZWVjaCB3YXMgY2FwdHVyZWQuIiwKICAgICAgICAia2V5X3BvaW50cyI6IGtleV9wb2ludHMsCiAgICAgICAgImtleV90ZXJtcyI6IFt7InRlcm0iOiB0LCAiZGVmaW5pdGlvbiI6IGRlZmluZSh0KX0gZm9yIHQgaW4gdGVybXNdLAogICAgICAgICJleGFtX3F1ZXN0aW9ucyI6IFtmIkV4cGxhaW4gJ3t0fScgaW4geW91ciBvd24gd29yZHMgYW5kIGdpdmUgYW4gZXhhbXBsZS4iIGZvciB0IGluIHRlcm1zWzo0XV0sCiAgICAgICAgInRvZG9zIjogdG9kb3NbOjhdLAogICAgICAgICJjb25mdXNpbmdfcGFydHMiOiBbXSwKICAgICAgICAiX25vdGUiOiAiTWFkZSBvZmZsaW5lIChubyBBSSBrZXkpLiBBZGQgYSBmcmVlIEdlbWluaSBrZXkgb3IgYSBDbGF1ZGUga2V5IGZvciBhIG11Y2ggYmV0dGVyIEFJLXdyaXR0ZW4gc2hlZXQuIiwKICAgIH0KCgojIC0tLS0tLS0tLS0tLS0tLS0gV3JpdGVycyAtLS0tLS0tLS0tLS0tLS0tCmRlZiB0b19tYXJrZG93bihzaGVldCwgY291cnNlLCBkYXRlKToKICAgIG1kID0gW2YiIyBTdHVkeSBzaGVldDoge3NoZWV0LmdldCgndGl0bGUnKSBvciBjb3Vyc2V9IiwKICAgICAgICAgIGYiKntjb3Vyc2V9IC0ge2RhdGUuc3RyZnRpbWUoJyVBICVCICVkLCAlWScpfSoiLCAiIl0KICAgIGlmIHNoZWV0LmdldCgiX25vdGUiKToKICAgICAgICBtZCArPSBbZiI+IHtzaGVldFsnX25vdGUnXX0iLCAiIl0KICAgIG1kICs9IFsiIyMgVEw7RFIiLCBzaGVldC5nZXQoInRsZHIiLCAiIiksICIiLCAiIyMgS2V5IHBvaW50cyJdCiAgICBtZCArPSBbZiItIHtwfSIgZm9yIHAgaW4gc2hlZXQuZ2V0KCJrZXlfcG9pbnRzIiwgW10pXQogICAgbWQgKz0gWyIiLCAiIyMgS2V5IHRlcm1zIl0KICAgIG1kICs9IFtmIi0gKip7dFsndGVybSddfSoqOiB7dFsnZGVmaW5pdGlvbiddfSIgZm9yIHQgaW4gc2hlZXQuZ2V0KCJrZXlfdGVybXMiLCBbXSldCiAgICBtZCArPSBbIiIsICIjIyBUZXN0IHlvdXJzZWxmIl0KICAgIG1kICs9IFtmIntpfS4ge3F9IiBmb3IgaSwgcSBpbiBlbnVtZXJhdGUoc2hlZXQuZ2V0KCJleGFtX3F1ZXN0aW9ucyIsIFtdKSwgMSldCiAgICBpZiBzaGVldC5nZXQoInRvZG9zIik6CiAgICAgICAgbWQgKz0gWyIiLCAiIyMgVG8tZG8iXQogICAgICAgIG1kICs9IFtmIi0gWyBdIHt0Wyd0YXNrJ119ICooZHVlIGluIH57dC5nZXQoJ2R1ZV9pbl9kYXlzJywgMyl9IGRheXMpKiIgZm9yIHQgaW4gc2hlZXRbInRvZG9zIl1dCiAgICBpZiBzaGVldC5nZXQoImNvbmZ1c2luZ19wYXJ0cyIpOgogICAgICAgIG1kICs9IFsiIiwgIiMjIEFzayB0aGUgcHJvZmVzc29yIC8gcmV2aWV3Il0KICAgICAgICBtZCArPSBbZiItIHtjfSIgZm9yIGMgaW4gc2hlZXRbImNvbmZ1c2luZ19wYXJ0cyJdXQogICAgbWQgKz0gWyIiLCAiIyMgUmV2aWV3IHNjaGVkdWxlIiwKICAgICAgICAgICAiLCAiLmpvaW4oKGRhdGUgKyB0aW1lZGVsdGEoZGF5cz1kKSkuc3RyZnRpbWUoIiVhICViICVkIikgZm9yIGQgaW4gY29uZmlnLlJFVklFV19EQVlTKV0KICAgIHJldHVybiAiXG4iLmpvaW4obWQpICsgIlxuIgoKCmRlZiB0b19kb2N4KHNoZWV0LCBjb3Vyc2UsIGRhdGUsIHBhdGgpOgogICAgdHJ5OgogICAgICAgIGZyb20gZG9jeCBpbXBvcnQgRG9jdW1lbnQKICAgICAgICBmcm9tIGRvY3guc2hhcmVkIGltcG9ydCBQdAogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHJldHVybiBOb25lCiAgICBkb2MgPSBEb2N1bWVudCgpCiAgICBkb2Muc3R5bGVzWyJOb3JtYWwiXS5mb250LnNpemUgPSBQdCgxMSkKICAgIGRvYy5hZGRfaGVhZGluZyhmIlN0dWR5IHNoZWV0OiB7c2hlZXQuZ2V0KCd0aXRsZScpIG9yIGNvdXJzZX0iLCAwKQogICAgZG9jLmFkZF9wYXJhZ3JhcGgoKS5hZGRfcnVuKGYie2NvdXJzZX0gLSB7ZGF0ZS5zdHJmdGltZSgnJUEgJUIgJWQsICVZJyl9IikuaXRhbGljID0gVHJ1ZQogICAgaWYgc2hlZXQuZ2V0KCJfbm90ZSIpOgogICAgICAgIGRvYy5hZGRfcGFyYWdyYXBoKHNoZWV0WyJfbm90ZSJdKS5ydW5zWzBdLml0YWxpYyA9IFRydWUKICAgIGRvYy5hZGRfaGVhZGluZygiVEw7RFIiLCAxKQogICAgZG9jLmFkZF9wYXJhZ3JhcGgoc2hlZXQuZ2V0KCJ0bGRyIiwgIiIpKQogICAgZG9jLmFkZF9oZWFkaW5nKCJLZXkgcG9pbnRzIiwgMSkKICAgIGZvciBwIGluIHNoZWV0LmdldCgia2V5X3BvaW50cyIsIFtdKToKICAgICAgICBkb2MuYWRkX3BhcmFncmFwaChwLCBzdHlsZT0iTGlzdCBCdWxsZXQiKQogICAgZG9jLmFkZF9oZWFkaW5nKCJLZXkgdGVybXMiLCAxKQogICAgZm9yIHQgaW4gc2hlZXQuZ2V0KCJrZXlfdGVybXMiLCBbXSk6CiAgICAgICAgcGFyYSA9IGRvYy5hZGRfcGFyYWdyYXBoKHN0eWxlPSJMaXN0IEJ1bGxldCIpCiAgICAgICAgcGFyYS5hZGRfcnVuKHRbInRlcm0iXSArICI6ICIpLmJvbGQgPSBUcnVlCiAgICAgICAgcGFyYS5hZGRfcnVuKHRbImRlZmluaXRpb24iXSkKICAgIGRvYy5hZGRfaGVhZGluZygiVGVzdCB5b3Vyc2VsZiIsIDEpCiAgICBmb3IgcSBpbiBzaGVldC5nZXQoImV4YW1fcXVlc3Rpb25zIiwgW10pOgogICAgICAgIGRvYy5hZGRfcGFyYWdyYXBoKHEsIHN0eWxlPSJMaXN0IE51bWJlciIpCiAgICBpZiBzaGVldC5nZXQoInRvZG9zIik6CiAgICAgICAgZG9jLmFkZF9oZWFkaW5nKCJUby1kbyIsIDEpCiAgICAgICAgZm9yIHQgaW4gc2hlZXRbInRvZG9zIl06CiAgICAgICAgICAgIGRvYy5hZGRfcGFyYWdyYXBoKGYi4piQIHt0Wyd0YXNrJ119IChkdWUgaW4gfnt0LmdldCgnZHVlX2luX2RheXMnLCAzKX0gZGF5cykiKQogICAgaWYgc2hlZXQuZ2V0KCJjb25mdXNpbmdfcGFydHMiKToKICAgICAgICBkb2MuYWRkX2hlYWRpbmcoIkFzayB0aGUgcHJvZmVzc29yIC8gcmV2aWV3IiwgMSkKICAgICAgICBmb3IgYyBpbiBzaGVldFsiY29uZnVzaW5nX3BhcnRzIl06CiAgICAgICAgICAgIGRvYy5hZGRfcGFyYWdyYXBoKGMsIHN0eWxlPSJMaXN0IEJ1bGxldCIpCiAgICBkb2Muc2F2ZShwYXRoKQogICAgcmV0dXJuIHBhdGgKCgpkZWYgX2ljc19lc2NhcGUoczogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gcy5yZXBsYWNlKCJcXCIsICJcXFxcIikucmVwbGFjZSgiOyIsICJcXDsiKS5yZXBsYWNlKCIsIiwgIlxcLCIpLnJlcGxhY2UoIlxuIiwgIlxcbiIpCgoKZGVmIHRvX2ljcyhzaGVldCwgY291cnNlLCBkYXRlLCBwYXRoKToKICAgIHRpdGxlID0gc2hlZXQuZ2V0KCJ0aXRsZSIpIG9yIGNvdXJzZQogICAgZXZlbnRzID0gW10KICAgIGZvciBkIGluIGNvbmZpZy5SRVZJRVdfREFZUzoKICAgICAgICBldmVudHMuYXBwZW5kKChkYXRlICsgdGltZWRlbHRhKGRheXM9ZCksIGYiUmV2aWV3OiB7dGl0bGV9IiwKICAgICAgICAgICAgICAgICAgICAgICBmIlNwYWNlZCByZXZpZXcgI3tjb25maWcuUkVWSUVXX0RBWVMuaW5kZXgoZCkgKyAxfS4gT3BlbiB5b3VyIHN0dWR5IHNoZWV0IGFuZCAiCiAgICAgICAgICAgICAgICAgICAgICAgZiJhbnN3ZXIgdGhlICdUZXN0IHlvdXJzZWxmJyBxdWVzdGlvbnMgd2l0aG91dCBsb29raW5nLiIsIDMwKSkKICAgIGZvciB0IGluIHNoZWV0LmdldCgidG9kb3MiLCBbXSk6CiAgICAgICAgZHVlID0gZGF0ZSArIHRpbWVkZWx0YShkYXlzPWludCh0LmdldCgiZHVlX2luX2RheXMiLCAzKSBvciAzKSkKICAgICAgICBldmVudHMuYXBwZW5kKChkdWUgLSB0aW1lZGVsdGEoZGF5cz0xKSwgZiJEdWUgc29vbiAoe2NvdXJzZX0pOiB7dFsndGFzayddWzo2MF19IiwgdFsidGFzayJdLCA2MCkpCiAgICBzdGFtcCA9IGRhdGV0aW1lLnV0Y25vdygpLnN0cmZ0aW1lKCIlWSVtJWRUJUglTSVTWiIpCiAgICBsaW5lcyA9IFsiQkVHSU46VkNBTEVOREFSIiwgIlZFUlNJT046Mi4wIiwgIlBST0RJRDotLy9UaGlyZEV5ZS8vRWFycGllY2UgUHJvdG90eXBlLy9FTiIsCiAgICAgICAgICAgICAiQ0FMU0NBTEU6R1JFR09SSUFOIiwgIk1FVEhPRDpQVUJMSVNIIl0KICAgIGZvciBkYXksIHN1bW1hcnksIGRlc2MsIG1pbnV0ZXMgaW4gZXZlbnRzOgogICAgICAgIHN0YXJ0ID0gZGF5LnJlcGxhY2UoaG91cj1jb25maWcuUkVNSU5ERVJfSE9VUiwgbWludXRlPTAsIHNlY29uZD0wLCBtaWNyb3NlY29uZD0wKQogICAgICAgIGVuZCA9IHN0YXJ0ICsgdGltZWRlbHRhKG1pbnV0ZXM9bWludXRlcykKICAgICAgICBsaW5lcyArPSBbIkJFR0lOOlZFVkVOVCIsIGYiVUlEOnt1dWlkLnV1aWQ0KCl9QHRoaXJkZXllIiwgZiJEVFNUQU1QOntzdGFtcH0iLAogICAgICAgICAgICAgICAgICBmIkRUU1RBUlQ6e3N0YXJ0LnN0cmZ0aW1lKCclWSVtJWRUJUglTSVTJyl9IiwgZiJEVEVORDp7ZW5kLnN0cmZ0aW1lKCclWSVtJWRUJUglTSVTJyl9IiwKICAgICAgICAgICAgICAgICAgZiJTVU1NQVJZOntfaWNzX2VzY2FwZShzdW1tYXJ5KX0iLCBmIkRFU0NSSVBUSU9OOntfaWNzX2VzY2FwZShkZXNjKX0iLAogICAgICAgICAgICAgICAgICAiQkVHSU46VkFMQVJNIiwgIkFDVElPTjpESVNQTEFZIiwgZiJERVNDUklQVElPTjp7X2ljc19lc2NhcGUoc3VtbWFyeSl9IiwKICAgICAgICAgICAgICAgICAgIlRSSUdHRVI6LVBUMTBNIiwgIkVORDpWQUxBUk0iLCAiRU5EOlZFVkVOVCJdCiAgICBsaW5lcy5hcHBlbmQoIkVORDpWQ0FMRU5EQVIiKQogICAgUGF0aChwYXRoKS53cml0ZV90ZXh0KCJcclxuIi5qb2luKGxpbmVzKSArICJcclxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHJldHVybiBwYXRoCgoKZGVmIGdlbmVyYXRlKHdyaXRlcikgLT4gZGljdDoKICAgICIiIkJ1aWxkIHRoZSBzdHVkeSBzaGVldCBmb3IgYSBmaW5pc2hlZCBsZWN0dXJlICh3cml0ZXIgPSBzeW5jLk5vdGVzV3JpdGVyKS4iIiIKICAgIHRyYW5zY3JpcHQgPSB3cml0ZXIudGltZWRfdHJhbnNjcmlwdCgpCiAgICBzbGlkZXMgPSB3cml0ZXIuc2xpZGVfdGV4dCgpCiAgICBzaGVldCA9IE5vbmUKICAgIGlmIGNvbmZpZy5haV9wcm92aWRlcigpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcHJpbnQoZiJbc3R1ZHkgc2hlZXRdIGFza2luZyB7Y29uZmlnLmFpX3Byb3ZpZGVyKCl9IHRvIHdyaXRlIHRoZSBzdHVkeSBzaGVldC4uLiIpCiAgICAgICAgICAgIHNoZWV0ID0gYWlfc2hlZXQod3JpdGVyLmNvdXJzZSwgdHJhbnNjcmlwdCwgc2xpZGVzKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcHJpbnQoZiJbc3R1ZHkgc2hlZXRdIEFJIGZhaWxlZCAoe2V9KTsgdXNpbmcgb2ZmbGluZSB2ZXJzaW9uIikKICAgIGlmIHNoZWV0IGlzIE5vbmU6CiAgICAgICAgc2hlZXQgPSBvZmZsaW5lX3NoZWV0KHdyaXRlci5jb3Vyc2UsIHdyaXRlci50cmFuc2NyaXB0KCksIHNsaWRlcykKCiAgICBkYXRlID0gd3JpdGVyLnN0YXJ0ZWQKICAgIG91dCA9IHsibWQiOiB3cml0ZXIuZm9sZGVyIC8gInN0dWR5X3NoZWV0Lm1kIiwgImljcyI6IHdyaXRlci5mb2xkZXIgLyAicmVtaW5kZXJzLmljcyJ9CiAgICBvdXRbIm1kIl0ud3JpdGVfdGV4dCh0b19tYXJrZG93bihzaGVldCwgd3JpdGVyLmNvdXJzZSwgZGF0ZSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0b19pY3Moc2hlZXQsIHdyaXRlci5jb3Vyc2UsIGRhdGUsIG91dFsiaWNzIl0pCiAgICBpZiB0b19kb2N4KHNoZWV0LCB3cml0ZXIuY291cnNlLCBkYXRlLCB3cml0ZXIuZm9sZGVyIC8gInN0dWR5X3NoZWV0LmRvY3giKToKICAgICAgICBvdXRbImRvY3giXSA9IHdyaXRlci5mb2xkZXIgLyAic3R1ZHlfc2hlZXQuZG9jeCIKICAgICh3cml0ZXIuZm9sZGVyIC8gInN0dWR5X3NoZWV0Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc2hlZXQsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHJldHVybiBvdXQK",
 "audio_capture.py": "IiIiCk1pY3JvcGhvbmUgLT4gYXVkaW8gY2h1bmtzLgoKTGl2ZU1pY1NvdXJjZSAgIDogcmVjb3JkcyBmcm9tIHRoZSBsYXB0b3AgbWljIChzdGFuZC1pbiBmb3IgdGhlIGVhcnBpZWNlIG1pYykuCkZpbGVBdWRpb1NvdXJjZSA6IHBsYXlzIGEgcmVjb3JkZWQgbGVjdHVyZSBmaWxlIGFzIGlmIGl0IHdlcmUgbGl2ZSAoZ3JlYXQgYmFja3VwIGZvciBkZW1vIGRheSkuCgpCb3RoIHB1dCAoc3RhcnRfdGltZV9zZWNvbmRzLCBudW1weV9mbG9hdDMyX2F1ZGlvKSBjaHVua3Mgb24gYHNlbGYuY2h1bmtzYCBmb3IgdGhlIHRyYW5zY3JpYmVyLAphbmQgYm90aCBvZmZlciBhc2tfcXVlc3Rpb24oKSwgd2hpY2ggZ3JhYnMgYSBmZXcgc2Vjb25kcyBvZiBZT1VSIHZvaWNlIGZvciB0aGUgYXNzaXN0YW50LgoiIiIKaW1wb3J0IHF1ZXVlCmltcG9ydCB0aHJlYWRpbmcKaW1wb3J0IHRpbWUKaW1wb3J0IHdhdmUKCmltcG9ydCBudW1weSBhcyBucAoKaW1wb3J0IGNvbmZpZwoKdHJ5OgogICAgaW1wb3J0IHNvdW5kZGV2aWNlIGFzIHNkCmV4Y2VwdCBFeGNlcHRpb246ICAjIHNvdW5kZGV2aWNlIG5vdCBpbnN0YWxsZWQgb3Igbm8gYXVkaW8gZGV2aWNlCiAgICBzZCA9IE5vbmUKCgpjbGFzcyBfQmFzZVNvdXJjZToKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLmNodW5rczogInF1ZXVlLlF1ZXVlW3R1cGxlW2Zsb2F0LCBucC5uZGFycmF5XV0iID0gcXVldWUuUXVldWUoKQogICAgICAgIHNlbGYucGF1c2VkID0gRmFsc2UKICAgICAgICBzZWxmLnNlc3Npb25fc3RhcnQgPSB0aW1lLnRpbWUoKQogICAgICAgIHNlbGYuX2J1ZiA9IFtdCiAgICAgICAgc2VsZi5fYnVmX2xlbiA9IDAKICAgICAgICBzZWxmLl9idWZfc3RhcnQgPSAwLjAKICAgICAgICBzZWxmLl9xdWVzdGlvbl9idWYgPSBOb25lCiAgICAgICAgc2VsZi5fbG9jayA9IHRocmVhZGluZy5Mb2NrKCkKICAgICAgICBzZWxmLmNodW5rX3NhbXBsZXMgPSBpbnQoY29uZmlnLkNIVU5LX1NFQ09ORFMgKiBjb25maWcuU0FNUExFX1JBVEUpCiAgICAgICAgc2VsZi5sZXZlbCA9IDAuMCAgICAgICAgICAjIGxpdmUgbG91ZG5lc3MgMC4uMSwgc2hvd24gYXMgdGhlIG1pYyBtZXRlciBpbiB0aGUgZGFzaGJvYXJkCgogICAgZGVmIF91cGRhdGVfbGV2ZWwoc2VsZiwgYmxvY2s6IG5wLm5kYXJyYXkpOgogICAgICAgIHJtcyA9IGZsb2F0KG5wLnNxcnQobnAubWVhbihibG9jayAqKiAyKSkpIGlmIGxlbihibG9jaykgZWxzZSAwLjAKICAgICAgICBzZWxmLmxldmVsID0gbWF4KG1pbigxLjAsIHJtcyAqIDEyKSwgc2VsZi5sZXZlbCAqIDAuOCkgICAjIHF1aWNrIHJpc2UsIHNtb290aCBmYWxsCgogICAgZGVmIGJlZ2luKHNlbGYpOgogICAgICAgICIiIlN0YXJ0IHRoZSBsZWN0dXJlIGNsb2NrIGFuZCBiZWdpbiByZWNvcmRpbmcuIiIiCiAgICAgICAgc2VsZi5zdGFydCgpCgogICAgZGVmIF9hZGRfbGVjdHVyZV9hdWRpbyhzZWxmLCBibG9jazogbnAubmRhcnJheSwgdDogZmxvYXQpOgogICAgICAgIGlmIHNlbGYuX2J1Zl9sZW4gPT0gMDoKICAgICAgICAgICAgc2VsZi5fYnVmX3N0YXJ0ID0gdAogICAgICAgIHNlbGYuX2J1Zi5hcHBlbmQoYmxvY2spCiAgICAgICAgc2VsZi5fYnVmX2xlbiArPSBsZW4oYmxvY2spCiAgICAgICAgaWYgc2VsZi5fYnVmX2xlbiA+PSBzZWxmLmNodW5rX3NhbXBsZXM6CiAgICAgICAgICAgIHNlbGYuX2ZsdXNoKCkKCiAgICBkZWYgX2ZsdXNoKHNlbGYpOgogICAgICAgIGlmIHNlbGYuX2J1Zl9sZW46CiAgICAgICAgICAgIHNlbGYuY2h1bmtzLnB1dCgoc2VsZi5fYnVmX3N0YXJ0LCBucC5jb25jYXRlbmF0ZShzZWxmLl9idWYpKSkKICAgICAgICBzZWxmLl9idWYsIHNlbGYuX2J1Zl9sZW4gPSBbXSwgMAoKICAgIGRlZiBhc2tfcXVlc3Rpb24oc2VsZikgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJSZWNvcmQgdGhlIHVzZXIncyBxdWVzdGlvbiBmcm9tIHRoZSBtaWNyb3Bob25lIChwdXNoLXRvLXRhbGspLiIiIgogICAgICAgIGlmIHNkIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigic291bmRkZXZpY2UgaXMgbm90IGluc3RhbGxlZCwgc28gdGhlIG1pYyBjYW4ndCBiZSB1c2VkIGZvciBxdWVzdGlvbnMuIikKICAgICAgICBhdWRpbyA9IHNkLnJlYyhpbnQoY29uZmlnLlFVRVNUSU9OX1NFQ09ORFMgKiBjb25maWcuU0FNUExFX1JBVEUpLAogICAgICAgICAgICAgICAgICAgICAgIHNhbXBsZXJhdGU9Y29uZmlnLlNBTVBMRV9SQVRFLCBjaGFubmVscz0xLCBkdHlwZT0iZmxvYXQzMiIpCiAgICAgICAgc2Qud2FpdCgpCiAgICAgICAgcmV0dXJuIGF1ZGlvWzosIDBdCgogICAgZGVmIHN0b3Aoc2VsZik6CiAgICAgICAgc2VsZi5fZmx1c2goKQoKCmNsYXNzIExpdmVNaWNTb3VyY2UoX0Jhc2VTb3VyY2UpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRldmljZT1Ob25lKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBzZCBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkluc3RhbGwgc291bmRkZXZpY2UgKHBpcCBpbnN0YWxsIHNvdW5kZGV2aWNlKSB0byB1c2UgdGhlIG1pY3JvcGhvbmUuIikKICAgICAgICBzZWxmLnN0cmVhbSA9IHNkLklucHV0U3RyZWFtKHNhbXBsZXJhdGU9Y29uZmlnLlNBTVBMRV9SQVRFLCBjaGFubmVscz0xLCBkdHlwZT0iZmxvYXQzMiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjYWxsYmFjaz1zZWxmLl9jYWxsYmFjaywgZGV2aWNlPWRldmljZSwgYmxvY2tzaXplPTE2MDApCgogICAgZGVmIF9jYWxsYmFjayhzZWxmLCBpbmRhdGEsIGZyYW1lcywgdGltZV9pbmZvLCBzdGF0dXMpOgogICAgICAgIGJsb2NrID0gaW5kYXRhWzosIDBdLmNvcHkoKQogICAgICAgIHNlbGYuX3VwZGF0ZV9sZXZlbChibG9jaykKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIGlmIHNlbGYuX3F1ZXN0aW9uX2J1ZiBpcyBub3QgTm9uZTogICAgICAgIyB1c2VyIGlzIGFza2luZyBhIHF1ZXN0aW9uCiAgICAgICAgICAgICAgICBzZWxmLl9xdWVzdGlvbl9idWYuYXBwZW5kKGJsb2NrKQogICAgICAgICAgICBlbGlmIG5vdCBzZWxmLnBhdXNlZDogICAgICAgICAgICAgICAgICAgICMgbm9ybWFsIGxlY3R1cmUgcmVjb3JkaW5nCiAgICAgICAgICAgICAgICBzZWxmLl9hZGRfbGVjdHVyZV9hdWRpbyhibG9jaywgdGltZS50aW1lKCkgLSBzZWxmLnNlc3Npb25fc3RhcnQpCgogICAgZGVmIHN0YXJ0KHNlbGYpOgogICAgICAgIHNlbGYuc2Vzc2lvbl9zdGFydCA9IHRpbWUudGltZSgpCiAgICAgICAgc2VsZi5zdHJlYW0uc3RhcnQoKQoKICAgIGRlZiBvcGVuX3BhdXNlZChzZWxmKToKICAgICAgICAiIiJPcGVuIHRoZSBtaWMgZm9yIHRoZSBzZXR1cCBzY3JlZW46IHRoZSBsZXZlbCBtZXRlciB3b3JrcywgYnV0IG5vdGhpbmcgaXMgcmVjb3JkZWQuIiIiCiAgICAgICAgc2VsZi5wYXVzZWQgPSBUcnVlCiAgICAgICAgc2VsZi5zdHJlYW0uc3RhcnQoKQoKICAgIGRlZiBiZWdpbihzZWxmKToKICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHNlbGYuX2J1Ziwgc2VsZi5fYnVmX2xlbiA9IFtdLCAwCiAgICAgICAgc2VsZi5zZXNzaW9uX3N0YXJ0ID0gdGltZS50aW1lKCkKICAgICAgICBzZWxmLnBhdXNlZCA9IEZhbHNlCgogICAgZGVmIGFza19xdWVzdGlvbihzZWxmKSAtPiBucC5uZGFycmF5OgogICAgICAgICMgUmV1c2UgdGhlIG9wZW4gc3RyZWFtIGluc3RlYWQgb2Ygb3BlbmluZyBhIHNlY29uZCBvbmUgKG1vcmUgcmVsaWFibGUgb24gbW9zdCBsYXB0b3BzKS4KICAgICAgICB3aXRoIHNlbGYuX2xvY2s6CiAgICAgICAgICAgIHNlbGYuX2ZsdXNoKCkKICAgICAgICAgICAgc2VsZi5fcXVlc3Rpb25fYnVmID0gW10KICAgICAgICB0aW1lLnNsZWVwKGNvbmZpZy5RVUVTVElPTl9TRUNPTkRTKQogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgYXVkaW8gPSBucC5jb25jYXRlbmF0ZShzZWxmLl9xdWVzdGlvbl9idWYpIGlmIHNlbGYuX3F1ZXN0aW9uX2J1ZiBlbHNlIG5wLnplcm9zKDEsIG5wLmZsb2F0MzIpCiAgICAgICAgICAgIHNlbGYuX3F1ZXN0aW9uX2J1ZiA9IE5vbmUKICAgICAgICByZXR1cm4gYXVkaW8KCiAgICBkZWYgc3RvcChzZWxmKToKICAgICAgICBzZWxmLnN0cmVhbS5zdG9wKCkKICAgICAgICBzZWxmLnN0cmVhbS5jbG9zZSgpCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBzZWxmLl9mbHVzaCgpCgoKZGVmIGxpc3RfaW5wdXRfZGV2aWNlcygpOgogICAgIiIiTWljcm9waG9uZXMgYXMgWyhkZXZpY2VfbnVtYmVyLCBuYW1lKV0sIHdpdGhvdXQgdGhlIGR1cGxpY2F0ZXMgV2luZG93cyBsaXN0cyBmb3IgZWFjaCBkcml2ZXIuIiIiCiAgICBpZiBzZCBpcyBOb25lOgogICAgICAgIHJldHVybiBbXQogICAgdHJ5OgogICAgICAgIGRldmljZXMgPSBzZC5xdWVyeV9kZXZpY2VzKCkKICAgICAgICBkZWZhdWx0X2luID0gc2QuZGVmYXVsdC5kZXZpY2VbMF0KICAgICAgICBob3N0YXBpID0gZGV2aWNlc1tkZWZhdWx0X2luXVsiaG9zdGFwaSJdIGlmIGRlZmF1bHRfaW4gaXMgbm90IE5vbmUgYW5kIGRlZmF1bHRfaW4gPj0gMCBlbHNlIDAKICAgICAgICBtaWNzLCBzZWVuID0gW10sIHNldCgpCiAgICAgICAgZm9yIGksIGQgaW4gZW51bWVyYXRlKGRldmljZXMpOgogICAgICAgICAgICBpZiBkWyJtYXhfaW5wdXRfY2hhbm5lbHMiXSA+IDAgYW5kIGRbImhvc3RhcGkiXSA9PSBob3N0YXBpIGFuZCBkWyJuYW1lIl0gbm90IGluIHNlZW46CiAgICAgICAgICAgICAgICBzZWVuLmFkZChkWyJuYW1lIl0pCiAgICAgICAgICAgICAgICBtaWNzLmFwcGVuZCgoaSwgZFsibmFtZSJdKSkKICAgICAgICByZXR1cm4gbWljcwogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gW10KCgpkZWYgbG9hZF9hdWRpbyhwYXRoOiBzdHIpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJMb2FkIGFueSBhdWRpbyBmaWxlIGFzIDE2IGtIeiBtb25vIGZsb2F0MzIuIiIiCiAgICB0cnk6ICAjIGZhc3Rlci13aGlzcGVyIHNoaXBzIGEgZGVjb2RlciB0aGF0IGhhbmRsZXMgbXAzL200YS9tcDQvd2F2CiAgICAgICAgZnJvbSBmYXN0ZXJfd2hpc3Blci5hdWRpbyBpbXBvcnQgZGVjb2RlX2F1ZGlvCiAgICAgICAgcmV0dXJuIGRlY29kZV9hdWRpbyhwYXRoLCBzYW1wbGluZ19yYXRlPWNvbmZpZy5TQU1QTEVfUkFURSkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcGFzcwogICAgdHJ5OgogICAgICAgIGltcG9ydCBzb3VuZGZpbGUgYXMgc2YKICAgICAgICBkYXRhLCBzciA9IHNmLnJlYWQocGF0aCwgZHR5cGU9ImZsb2F0MzIiLCBhbHdheXNfMmQ9VHJ1ZSkKICAgICAgICBkYXRhID0gZGF0YS5tZWFuKGF4aXM9MSkKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICB3aXRoIHdhdmUub3BlbihwYXRoLCAicmIiKSBhcyB3OiAgIyBwbGFpbiAud2F2IGZhbGxiYWNrLCBubyBleHRyYSBwYWNrYWdlcwogICAgICAgICAgICBzciwgY2gsIHdpZHRoID0gdy5nZXRmcmFtZXJhdGUoKSwgdy5nZXRuY2hhbm5lbHMoKSwgdy5nZXRzYW1wd2lkdGgoKQogICAgICAgICAgICByYXcgPSB3LnJlYWRmcmFtZXMody5nZXRuZnJhbWVzKCkpCiAgICAgICAgZHR5cGUgPSB7MTogbnAudWludDgsIDI6IG5wLmludDE2LCA0OiBucC5pbnQzMn1bd2lkdGhdCiAgICAgICAgZGF0YSA9IG5wLmZyb21idWZmZXIocmF3LCBkdHlwZT1kdHlwZSkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgaWYgd2lkdGggPT0gMToKICAgICAgICAgICAgZGF0YSA9IChkYXRhIC0gMTI4KSAvIDEyOAogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGRhdGEgLz0gZmxvYXQobnAuaWluZm8oZHR5cGUpLm1heCkKICAgICAgICBkYXRhID0gZGF0YS5yZXNoYXBlKC0xLCBjaCkubWVhbihheGlzPTEpCiAgICBpZiBzciAhPSBjb25maWcuU0FNUExFX1JBVEU6ICAjIHNpbXBsZSByZXNhbXBsZQogICAgICAgIG4gPSBpbnQobGVuKGRhdGEpICogY29uZmlnLlNBTVBMRV9SQVRFIC8gc3IpCiAgICAgICAgZGF0YSA9IG5wLmludGVycChucC5saW5zcGFjZSgwLCBsZW4oZGF0YSkgLSAxLCBuKSwgbnAuYXJhbmdlKGxlbihkYXRhKSksIGRhdGEpLmFzdHlwZShucC5mbG9hdDMyKQogICAgcmV0dXJuIGRhdGEKCgpjbGFzcyBGaWxlQXVkaW9Tb3VyY2UoX0Jhc2VTb3VyY2UpOgogICAgIiIiRmVlZHMgYSByZWNvcmRlZCBsZWN0dXJlIGluIHJlYWwgdGltZSwgYW5kIHBsYXlzIGl0IG91dCBsb3VkIHNvIHRoZSBhdWRpZW5jZSBoZWFycyBpdC4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcGF0aDogc3RyLCBwbGF5X291dF9sb3VkOiBib29sID0gVHJ1ZSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5hdWRpbyA9IGxvYWRfYXVkaW8ocGF0aCkKICAgICAgICBzZWxmLnBsYXkgPSBwbGF5X291dF9sb3VkIGFuZCBzZCBpcyBub3QgTm9uZQogICAgICAgIHNlbGYuZmluaXNoZWQgPSB0aHJlYWRpbmcuRXZlbnQoKQogICAgICAgIHNlbGYuX3N0b3BwZWQgPSBGYWxzZQogICAgICAgIHNlbGYuX3RocmVhZCA9IHRocmVhZGluZy5UaHJlYWQodGFyZ2V0PXNlbGYuX3J1biwgZGFlbW9uPVRydWUpCgogICAgZGVmIHN0YXJ0KHNlbGYpOgogICAgICAgIHNlbGYuc2Vzc2lvbl9zdGFydCA9IHRpbWUudGltZSgpCiAgICAgICAgc2VsZi5fdGhyZWFkLnN0YXJ0KCkKCiAgICBkZWYgX3J1bihzZWxmKToKICAgICAgICBibG9jayA9IGNvbmZpZy5TQU1QTEVfUkFURSAvLyAxMCAgIyAwLjEgcwogICAgICAgIHBvcyA9IDAKICAgICAgICBvdXQgPSBOb25lCiAgICAgICAgaWYgc2VsZi5wbGF5OgogICAgICAgICAgICBvdXQgPSBzZC5PdXRwdXRTdHJlYW0oc2FtcGxlcmF0ZT1jb25maWcuU0FNUExFX1JBVEUsIGNoYW5uZWxzPTEsIGR0eXBlPSJmbG9hdDMyIikKICAgICAgICAgICAgb3V0LnN0YXJ0KCkKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgd2hpbGUgcG9zIDwgbGVuKHNlbGYuYXVkaW8pIGFuZCBub3Qgc2VsZi5fc3RvcHBlZDoKICAgICAgICAgICAgaWYgc2VsZi5wYXVzZWQ6CiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDAuMSkKICAgICAgICAgICAgICAgIHQwICs9IDAuMQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcGllY2UgPSBzZWxmLmF1ZGlvW3Bvczpwb3MgKyBibG9ja10KICAgICAgICAgICAgaWYgb3V0IGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgb3V0LndyaXRlKHBpZWNlLnJlc2hhcGUoLTEsIDEpKSAgICAgICMgYmxvY2tzIH4wLjEgcywga2VlcHMgdXMgcmVhbC10aW1lCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKG1heCgwLjAsIHQwICsgKHBvcyArIGxlbihwaWVjZSkpIC8gY29uZmlnLlNBTVBMRV9SQVRFIC0gdGltZS50aW1lKCkpKQogICAgICAgICAgICBzZWxmLl91cGRhdGVfbGV2ZWwocGllY2UpCiAgICAgICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgICAgIHNlbGYuX2FkZF9sZWN0dXJlX2F1ZGlvKHBpZWNlLCBwb3MgLyBjb25maWcuU0FNUExFX1JBVEUpCiAgICAgICAgICAgIHBvcyArPSBibG9jawogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgc2VsZi5fZmx1c2goKQogICAgICAgIGlmIG91dCBpcyBub3QgTm9uZToKICAgICAgICAgICAgb3V0LnN0b3AoKQogICAgICAgICAgICBvdXQuY2xvc2UoKQogICAgICAgIHNlbGYuZmluaXNoZWQuc2V0KCkKCiAgICBkZWYgc3RvcChzZWxmKToKICAgICAgICBzZWxmLl9zdG9wcGVkID0gVHJ1ZQogICAgICAgIHNlbGYuZmluaXNoZWQud2FpdCh0aW1lb3V0PTIpCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBzZWxmLl9mbHVzaCgpCg==",
 "transcriber.py": "IiIiClNwZWVjaCAtPiB0ZXh0LCBmdWxseSBvZmZsaW5lLCB1c2luZyBmYXN0ZXItd2hpc3BlciAoZmFsbHMgYmFjayB0byBvcGVuYWktd2hpc3BlcikuClRoZSBmaXJzdCBydW4gZG93bmxvYWRzIHRoZSBtb2RlbCBvbmNlICh+NzUtMTUwIE1CIGZvciBiYXNlLmVuKTsgYWZ0ZXIgdGhhdCBubyBpbnRlcm5ldCBpcyBuZWVkZWQuCiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKCmltcG9ydCBjb25maWcKCgpjbGFzcyBUcmFuc2NyaWJlcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbF9zaXplOiBzdHIgPSBjb25maWcuV0hJU1BFUl9NT0RFTCk6CiAgICAgICAgc2VsZi5iYWNrZW5kID0gTm9uZQogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBmYXN0ZXJfd2hpc3BlciBpbXBvcnQgV2hpc3Blck1vZGVsCiAgICAgICAgICAgIHByaW50KGYiW3RyYW5zY3JpYmVyXSBsb2FkaW5nIGZhc3Rlci13aGlzcGVyICd7bW9kZWxfc2l6ZX0nIChmaXJzdCBydW4gZG93bmxvYWRzIGl0KS4uLiIpCiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBXaGlzcGVyTW9kZWwobW9kZWxfc2l6ZSwgZGV2aWNlPSJjcHUiLCBjb21wdXRlX3R5cGU9ImludDgiKQogICAgICAgICAgICBzZWxmLmJhY2tlbmQgPSAiZmFzdGVyLXdoaXNwZXIiCiAgICAgICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBpbXBvcnQgd2hpc3BlcgogICAgICAgICAgICAgICAgcHJpbnQoZiJbdHJhbnNjcmliZXJdIGxvYWRpbmcgb3BlbmFpLXdoaXNwZXIgJ3ttb2RlbF9zaXplfScuLi4iKQogICAgICAgICAgICAgICAgc2VsZi5tb2RlbCA9IHdoaXNwZXIubG9hZF9tb2RlbChtb2RlbF9zaXplKQogICAgICAgICAgICAgICAgc2VsZi5iYWNrZW5kID0gIndoaXNwZXIiCiAgICAgICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiSW5zdGFsbCBmYXN0ZXItd2hpc3BlcjogIHBpcCBpbnN0YWxsIGZhc3Rlci13aGlzcGVyIikKICAgICAgICBwcmludChmIlt0cmFuc2NyaWJlcl0gcmVhZHkgKHtzZWxmLmJhY2tlbmR9KSIpCiAgICAgICAgc2VsZi5fcHJldl90ZXh0ID0gIiIKCiAgICBkZWYgdHJhbnNjcmliZShzZWxmLCBhdWRpbzogbnAubmRhcnJheSwgdXNlX2NvbnRleHQ6IGJvb2wgPSBUcnVlKSAtPiBzdHI6CiAgICAgICAgYXVkaW8gPSBhdWRpby5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICBpZiBsZW4oYXVkaW8pID09IDAgb3IgZmxvYXQobnAuc3FydChucC5tZWFuKGF1ZGlvICoqIDIpKSkgPCBjb25maWcuU0lMRU5DRV9STVM6CiAgICAgICAgICAgIHJldHVybiAiIiAgIyBzaWxlbmNlOiBza2lwIChzYXZlcyBDUFUgYW5kIGF2b2lkcyBXaGlzcGVyICJoYWxsdWNpbmF0aW5nIiB3b3JkcykKICAgICAgICAjIFBhc3NpbmcgdGhlIHByZXZpb3VzIHNlbnRlbmNlIGFzIGEgcHJvbXB0IGhlbHBzIFdoaXNwZXIga2VlcCBuYW1lcy90ZXJtcyBjb25zaXN0ZW50LgogICAgICAgIHByb21wdCA9IHNlbGYuX3ByZXZfdGV4dFstMjAwOl0gaWYgdXNlX2NvbnRleHQgZWxzZSBOb25lCiAgICAgICAgaWYgc2VsZi5iYWNrZW5kID09ICJmYXN0ZXItd2hpc3BlciI6CiAgICAgICAgICAgIHNlZ21lbnRzLCBfID0gc2VsZi5tb2RlbC50cmFuc2NyaWJlKGF1ZGlvLCBsYW5ndWFnZT0iZW4iLCBiZWFtX3NpemU9MSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdmFkX2ZpbHRlcj1UcnVlLCBpbml0aWFsX3Byb21wdD1wcm9tcHQpCiAgICAgICAgICAgIHRleHQgPSAiICIuam9pbihzLnRleHQuc3RyaXAoKSBmb3IgcyBpbiBzZWdtZW50cykuc3RyaXAoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJlc3VsdCA9IHNlbGYubW9kZWwudHJhbnNjcmliZShhdWRpbywgbGFuZ3VhZ2U9ImVuIiwgZnAxNj1GYWxzZSwgaW5pdGlhbF9wcm9tcHQ9cHJvbXB0KQogICAgICAgICAgICB0ZXh0ID0gcmVzdWx0WyJ0ZXh0Il0uc3RyaXAoKQogICAgICAgIGlmIHVzZV9jb250ZXh0IGFuZCB0ZXh0OgogICAgICAgICAgICBzZWxmLl9wcmV2X3RleHQgPSB0ZXh0CiAgICAgICAgcmV0dXJuIHRleHQK",
 "colab_helpers.py": "IiIiCkhlbHBlcnMgZm9yIHJ1bm5pbmcgVGhpcmQgRXllIGluIEdvb2dsZSBDb2xhYi4KCkNvbGFiIHJ1bnMgb24gYSBHb29nbGUgc2VydmVyLCBzbyBpdCBjYW4ndCB1c2UgeW91ciB3ZWJjYW0vbWljIGxpdmUuIEluc3RlYWQgaXQgd29ya3Mgb24gYQpsZWN0dXJlIHlvdSB1cGxvYWQgKGF1ZGlvIG9yIHZpZGVvKSwgYW5kIHlvdSBhc2sgcXVlc3Rpb25zIGJ5IHR5cGluZyBvciB0aHJvdWdoIHRoZSBicm93c2VyIG1pYy4KSXQgcmV1c2VzIHRoZSBzYW1lIG1vZHVsZXMgYXMgdGhlIGxhcHRvcCB2ZXJzaW9uOiBzeW5jLnB5LCB2aXNpb24ucHksIGFzc2lzdGFudC5weSwgc3R1ZHlfc2hlZXQucHkuCiIiIgppbXBvcnQgYmFzZTY0CmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKCmltcG9ydCBjb25maWcKZnJvbSBhc3Npc3RhbnQgaW1wb3J0IEFzc2lzdGFudApmcm9tIHN5bmMgaW1wb3J0IE5vdGVzV3JpdGVyLCBmbXRfdGltZQpmcm9tIHZpc2lvbiBpbXBvcnQgX3NtYWxsX2dyYXksIGNoYW5nZV9wZXJjZW50LCBvY3JfaW1hZ2UKClZJREVPX0VYVFMgPSB7Ii5tcDQiLCAiLm1vdiIsICIubWt2IiwgIi53ZWJtIiwgIi5hdmkiLCAiLm00diJ9Cl9tb2RlbHMgPSB7fQoKCiMgLS0tLS0tLS0tLS0tLS0tLSBzcGVlY2ggdG8gdGV4dCAtLS0tLS0tLS0tLS0tLS0tCmRlZiBfbG9hZF9tb2RlbChzaXplOiBzdHIsIHVzZV9ncHU6IGJvb2wpOgogICAgZnJvbSBmYXN0ZXJfd2hpc3BlciBpbXBvcnQgV2hpc3Blck1vZGVsCiAgICBrZXkgPSAoc2l6ZSwgdXNlX2dwdSkKICAgIGlmIGtleSBub3QgaW4gX21vZGVsczoKICAgICAgICBpZiB1c2VfZ3B1OgogICAgICAgICAgICBfbW9kZWxzW2tleV0gPSBXaGlzcGVyTW9kZWwoc2l6ZSwgZGV2aWNlPSJjdWRhIiwgY29tcHV0ZV90eXBlPSJmbG9hdDE2IikKICAgICAgICBlbHNlOgogICAgICAgICAgICBfbW9kZWxzW2tleV0gPSBXaGlzcGVyTW9kZWwoc2l6ZSwgZGV2aWNlPSJjcHUiLCBjb21wdXRlX3R5cGU9ImludDgiKQogICAgcmV0dXJuIF9tb2RlbHNba2V5XQoKCmRlZiBfZ3B1X2F2YWlsYWJsZSgpIC0+IGJvb2w6CiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgcmV0dXJuIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIHRyYW5zY3JpYmVfZmlsZShwYXRoOiBzdHIsIHdyaXRlcjogTm90ZXNXcml0ZXIsIG1vZGVsX3NpemU6IHN0ciA9ICJiYXNlLmVuIiwgdXNlX2dwdT1Ob25lKSAtPiBpbnQ6CiAgICAiIiJUcmFuc2NyaWJlIGEgd2hvbGUgbGVjdHVyZSBmaWxlIGludG8gdGhlIG5vdGVzLCBwcmludGluZyBsaW5lcyBhcyB0aGV5IGNvbWUuIFJldHVybnMgbGluZSBjb3VudC4iIiIKICAgIGlmIHVzZV9ncHUgaXMgTm9uZToKICAgICAgICB1c2VfZ3B1ID0gX2dwdV9hdmFpbGFibGUoKQogICAgZm9yIGF0dGVtcHRfZ3B1IGluIChbVHJ1ZSwgRmFsc2VdIGlmIHVzZV9ncHUgZWxzZSBbRmFsc2VdKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIG1vZGVsID0gX2xvYWRfbW9kZWwobW9kZWxfc2l6ZSwgYXR0ZW1wdF9ncHUpCiAgICAgICAgICAgIHByaW50KGYiVHJhbnNjcmliaW5nIG9uIHsnR1BVJyBpZiBhdHRlbXB0X2dwdSBlbHNlICdDUFUnfSB3aXRoICd7bW9kZWxfc2l6ZX0nLi4uIikKICAgICAgICAgICAgc2VnbWVudHMsIGluZm8gPSBtb2RlbC50cmFuc2NyaWJlKHBhdGgsIGxhbmd1YWdlPSJlbiIsIHZhZF9maWx0ZXI9VHJ1ZSwgYmVhbV9zaXplPTEpCiAgICAgICAgICAgIHRvdGFsID0gaW5mby5kdXJhdGlvbiBvciAwCiAgICAgICAgICAgIHN0YXJ0X2Nsb2NrLCBjb3VudCA9IHRpbWUudGltZSgpLCAwCiAgICAgICAgICAgIGZvciBzZWcgaW4gc2VnbWVudHM6CiAgICAgICAgICAgICAgICB0ZXh0ID0gc2VnLnRleHQuc3RyaXAoKQogICAgICAgICAgICAgICAgaWYgbm90IHRleHQ6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHdyaXRlci5hZGRfc3BlZWNoKHNlZy5zdGFydCwgdGV4dCkKICAgICAgICAgICAgICAgIGNvdW50ICs9IDEKICAgICAgICAgICAgICAgIHBjdCA9IGYiezEwMCAqIHNlZy5lbmQgLyB0b3RhbDo0LjBmfSUiIGlmIHRvdGFsIGVsc2UgIiIKICAgICAgICAgICAgICAgIHByaW50KGYie3BjdH0gIHtmbXRfdGltZShzZWcuc3RhcnQpfSAge3RleHR9IikKICAgICAgICAgICAgcHJpbnQoZiJcbkRvbmU6IHtjb3VudH0gbGluZXMgZnJvbSB7Zm10X3RpbWUodG90YWwpfSBvZiBhdWRpbyAiCiAgICAgICAgICAgICAgICAgIGYiaW4ge2ZtdF90aW1lKHRpbWUudGltZSgpIC0gc3RhcnRfY2xvY2spfS4iKQogICAgICAgICAgICByZXR1cm4gY291bnQKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGlmIGF0dGVtcHRfZ3B1OgogICAgICAgICAgICAgICAgcHJpbnQoZiIoR1BVIGRpZG4ndCB3b3JrOiB7ZS5fX2NsYXNzX18uX19uYW1lX199LiBTd2l0Y2hpbmcgdG8gQ1BVLikiKQogICAgICAgICAgICAgICAgd2l0aCB3cml0ZXIuX2xvY2s6ICAgIyB0aHJvdyBhd2F5IGFueSBwYXJ0aWFsIEdQVSBsaW5lcyBiZWZvcmUgcmV0cnlpbmcKICAgICAgICAgICAgICAgICAgICB3cml0ZXIuZW50cmllcyA9IFt4IGZvciB4IGluIHdyaXRlci5lbnRyaWVzIGlmIHhbMV0gIT0gInNwZWVjaCJdCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICByYWlzZQogICAgcmV0dXJuIDAKCgpkZWYgdHJhbnNjcmliZV9jbGlwKHBhdGg6IHN0ciwgbW9kZWxfc2l6ZTogc3RyID0gImJhc2UuZW4iKSAtPiBzdHI6CiAgICAiIiJTaG9ydCBjbGlwIChlLmcuIGEgc3Bva2VuIHF1ZXN0aW9uKSAtPiB0ZXh0LiIiIgogICAgbW9kZWwgPSBfbG9hZF9tb2RlbChtb2RlbF9zaXplLCBGYWxzZSkKICAgIHNlZ21lbnRzLCBfID0gbW9kZWwudHJhbnNjcmliZShwYXRoLCBsYW5ndWFnZT0iZW4iLCBiZWFtX3NpemU9MSkKICAgIHJldHVybiAiICIuam9pbihzLnRleHQuc3RyaXAoKSBmb3IgcyBpbiBzZWdtZW50cykuc3RyaXAoKQoKCiMgLS0tLS0tLS0tLS0tLS0tLSBUaGlyZCBFeWUgY2FtZXJhOiBzbGlkZXMgLS0tLS0tLS0tLS0tLS0tLQpkZWYgZXh0cmFjdF9zbGlkZXModmlkZW9fcGF0aDogc3RyLCB3cml0ZXI6IE5vdGVzV3JpdGVyLCBldmVyeV9zZWM6IGZsb2F0ID0gMS4wKSAtPiBpbnQ6CiAgICAiIiJGaW5kIHNsaWRlL2JvYXJkIGNoYW5nZXMgaW4gYSBsZWN0dXJlIHZpZGVvLCBzYXZlIHNuYXBzaG90cyBhbmQgcmVhZCB0aGVpciB0ZXh0LiIiIgogICAgaWYgUGF0aCh2aWRlb19wYXRoKS5zdWZmaXgubG93ZXIoKSBub3QgaW4gVklERU9fRVhUUzoKICAgICAgICBwcmludCgiVGhpcyBpcyBhbiBhdWRpbyBmaWxlLCBzbyB0aGVyZSBhcmUgbm8gc2xpZGVzIHRvIGNhcHR1cmUuIChVcGxvYWQgYSB2aWRlbyB0byB1c2UgdGhlIGNhbWVyYSBmZWF0dXJlLikiKQogICAgICAgIHJldHVybiAwCiAgICBjYXAgPSBjdjIuVmlkZW9DYXB0dXJlKHZpZGVvX3BhdGgpCiAgICBpZiBub3QgY2FwLmlzT3BlbmVkKCk6CiAgICAgICAgcHJpbnQoIkNvdWxkbid0IG9wZW4gdGhlIHZpZGVvIGZvciBzbGlkZSBjYXB0dXJlLiIpCiAgICAgICAgcmV0dXJuIDAKICAgIGZwcyA9IGNhcC5nZXQoY3YyLkNBUF9QUk9QX0ZQUykgb3IgMzAKICAgIHN0ZXAgPSBtYXgoMSwgaW50KGZwcyAqIGV2ZXJ5X3NlYykpCiAgICBwcmV2LCBsYXN0X3NhdmVkLCBzdGFibGUsIHNhdmVkLCBpZHggPSBOb25lLCBOb25lLCAwLCAwLCAwCiAgICBwcmludCgiU2Nhbm5pbmcgdmlkZW8gZm9yIHNsaWRlIGNoYW5nZXMuLi4iKQogICAgd2hpbGUgVHJ1ZToKICAgICAgICBpZiBpZHggJSBzdGVwOiAgICAgICAgICAgICAgICAgICAgICAjIHNraXAgZnJhbWVzIHF1aWNrbHkgd2l0aG91dCBkZWNvZGluZyB0aGVtCiAgICAgICAgICAgIGlmIG5vdCBjYXAuZ3JhYigpOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWR4ICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBvaywgZnJhbWUgPSBjYXAucmVhZCgpCiAgICAgICAgaWYgbm90IG9rOgogICAgICAgICAgICBicmVhawogICAgICAgIHQgPSBpZHggLyBmcHMKICAgICAgICBnID0gX3NtYWxsX2dyYXkoZnJhbWUpCiAgICAgICAgbW92aW5nID0gcHJldiBpcyBub3QgTm9uZSBhbmQgY2hhbmdlX3BlcmNlbnQoZywgcHJldikgPiAxLjAKICAgICAgICBwcmV2ID0gZwogICAgICAgIHN0YWJsZSA9IDAgaWYgbW92aW5nIGVsc2Ugc3RhYmxlICsgMQogICAgICAgIGNoYW5nZWQgPSBsYXN0X3NhdmVkIGlzIE5vbmUgb3IgY2hhbmdlX3BlcmNlbnQoZywgbGFzdF9zYXZlZCkgPiBjb25maWcuU0xJREVfQ0hBTkdFX1RIUkVTSE9MRAogICAgICAgIGlmIGNoYW5nZWQgYW5kIHN0YWJsZSA+PSAyOgogICAgICAgICAgICBsYXN0X3NhdmVkID0gZwogICAgICAgICAgICBzYXZlZCArPSAxCiAgICAgICAgICAgIHBhdGggPSB3cml0ZXIuZm9sZGVyIC8gZiJzbGlkZV97c2F2ZWQ6MDNkfS5qcGciCiAgICAgICAgICAgIGN2Mi5pbXdyaXRlKHN0cihwYXRoKSwgZnJhbWUpCiAgICAgICAgICAgIHRleHQgPSBvY3JfaW1hZ2UoZnJhbWUpCiAgICAgICAgICAgIHdyaXRlci5hZGRfc2xpZGUodCwgc3RyKHBhdGgpLCB0ZXh0KQogICAgICAgICAgICBwcmludChmIiAge2ZtdF90aW1lKHQpfSAgc2xpZGUge3NhdmVkfToge3RleHRbOjcwXS5yZXBsYWNlKGNocigxMCksICcgLyAnKSBvciAnKG5vIHRleHQgZm91bmQpJ30iKQogICAgICAgIGlkeCArPSAxCiAgICBjYXAucmVsZWFzZSgpCiAgICBwcmludChmIkNhcHR1cmVkIHtzYXZlZH0gc2xpZGVzLiIpCiAgICByZXR1cm4gc2F2ZWQKCgpkZWYgYWRkX3NsaWRlX3Bob3RvcyhwYXRocywgd3JpdGVyOiBOb3Rlc1dyaXRlcikgLT4gaW50OgogICAgIiIiQWRkIHBob3RvcyBvZiBzbGlkZXMvd2hpdGVib2FyZCB5b3UgdG9vayB5b3Vyc2VsZi4iIiIKICAgIG4gPSAwCiAgICBlbmRfdCA9IHdyaXRlci5lbnRyaWVzWy0xXVswXSArIDEgaWYgd3JpdGVyLmVudHJpZXMgZWxzZSAwCiAgICBmb3IgcCBpbiBwYXRoczoKICAgICAgICBpbWcgPSBjdjIuaW1yZWFkKHN0cihwKSkKICAgICAgICBpZiBpbWcgaXMgTm9uZToKICAgICAgICAgICAgcHJpbnQoZiIgIHNraXBwZWQge3B9IChub3QgYW4gaW1hZ2UpIikKICAgICAgICAgICAgY29udGludWUKICAgICAgICBuICs9IDEKICAgICAgICBkZXN0ID0gd3JpdGVyLmZvbGRlciAvIGYicGhvdG9fe246MDNkfS5qcGciCiAgICAgICAgY3YyLmltd3JpdGUoc3RyKGRlc3QpLCBpbWcpCiAgICAgICAgdGV4dCA9IG9jcl9pbWFnZShpbWcpCiAgICAgICAgd3JpdGVyLmFkZF9zbGlkZShlbmRfdCwgc3RyKGRlc3QpLCB0ZXh0KQogICAgICAgIHByaW50KGYiICB7UGF0aChwKS5uYW1lfToge3RleHRbOjgwXS5yZXBsYWNlKGNocigxMCksICcgLyAnKSBvciAnKG5vIHRleHQgZm91bmQpJ30iKQogICAgcmV0dXJuIG4KCgojIC0tLS0tLS0tLS0tLS0tLS0gYXNzaXN0YW50IC0tLS0tLS0tLS0tLS0tLS0KZGVmIGFzayhxdWVzdGlvbjogc3RyLCB3cml0ZXI6IE5vdGVzV3JpdGVyLCBzcGVha19hbnN3ZXI6IGJvb2wgPSBUcnVlKSAtPiBzdHI6CiAgICBhc3Npc3RhbnQgPSBBc3Npc3RhbnQoKQogICAgYW5zd2VyID0gYXNzaXN0YW50LmFuc3dlcihxdWVzdGlvbiwgd3JpdGVyLnRyYW5zY3JpcHQoKSwgd3JpdGVyLnNsaWRlX3RleHQoKSkKICAgIHQgPSB3cml0ZXIuZW50cmllc1stMV1bMF0gaWYgd3JpdGVyLmVudHJpZXMgZWxzZSAwCiAgICB3cml0ZXIuYWRkX3FhKHQsIHF1ZXN0aW9uLCBhbnN3ZXIpCiAgICBwcmludChmIllPVToge3F1ZXN0aW9ufVxuQUk6ICB7YW5zd2VyfSIpCiAgICBpZiBzcGVha19hbnN3ZXI6CiAgICAgICAgc2F5X2luX2Jyb3dzZXIoYW5zd2VyKQogICAgcmV0dXJuIGFuc3dlcgoKCmRlZiBzYXlfaW5fYnJvd3Nlcih0ZXh0OiBzdHIpOgogICAgIiIiUGxheSB0aGUgYW5zd2VyIGFzIGF1ZGlvIGluIHRoZSBub3RlYm9vayAobmVlZHMgaW50ZXJuZXQsIHdoaWNoIENvbGFiIGhhcykuIiIiCiAgICB0cnk6CiAgICAgICAgZnJvbSBndHRzIGltcG9ydCBnVFRTCiAgICAgICAgZnJvbSBJUHl0aG9uLmRpc3BsYXkgaW1wb3J0IEF1ZGlvLCBkaXNwbGF5CiAgICAgICAgb3V0ID0gIi9jb250ZW50L2Fuc3dlci5tcDMiIGlmIFBhdGgoIi9jb250ZW50IikuZXhpc3RzKCkgZWxzZSAiYW5zd2VyLm1wMyIKICAgICAgICBnVFRTKHRleHQpLnNhdmUob3V0KQogICAgICAgIGRpc3BsYXkoQXVkaW8ob3V0LCBhdXRvcGxheT1UcnVlKSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIihjb3VsZG4ndCBwbGF5IGF1ZGlvOiB7ZX0pIikKCgpSRUNPUkRfSlMgPSAiIiIKYXN5bmMgZnVuY3Rpb24gcmVjb3JkUXVlc3Rpb24obXMpIHsKICBjb25zdCBzdHJlYW0gPSBhd2FpdCBuYXZpZ2F0b3IubWVkaWFEZXZpY2VzLmdldFVzZXJNZWRpYSh7YXVkaW86IHRydWV9KTsKICBjb25zdCByZWMgPSBuZXcgTWVkaWFSZWNvcmRlcihzdHJlYW0pOwogIGNvbnN0IGNodW5rcyA9IFtdOwogIHJlYy5vbmRhdGFhdmFpbGFibGUgPSBlID0+IGNodW5rcy5wdXNoKGUuZGF0YSk7CiAgY29uc3Qgc3RvcHBlZCA9IG5ldyBQcm9taXNlKHJlcyA9PiByZWMub25zdG9wID0gcmVzKTsKICBjb25zdCBib3ggPSBkb2N1bWVudC5jcmVhdGVFbGVtZW50KCdkaXYnKTsKICBib3guc3R5bGUgPSAncGFkZGluZzo4cHggMTJweDtiYWNrZ3JvdW5kOiNmZmEwMDA7Y29sb3I6IzAwMDtib3JkZXItcmFkaXVzOjhweDtmb250LXdlaWdodDpib2xkO2Rpc3BsYXk6aW5saW5lLWJsb2NrJzsKICBib3gudGV4dENvbnRlbnQgPSAn4pePIExpc3RlbmluZy4uLiBhc2sgeW91ciBxdWVzdGlvbiBub3cnOwogIGRvY3VtZW50LmJvZHkuYXBwZW5kQ2hpbGQoYm94KTsKICByZWMuc3RhcnQoKTsKICBhd2FpdCBuZXcgUHJvbWlzZShyID0+IHNldFRpbWVvdXQociwgbXMpKTsKICByZWMuc3RvcCgpOyBhd2FpdCBzdG9wcGVkOwogIHN0cmVhbS5nZXRUcmFja3MoKS5mb3JFYWNoKHQgPT4gdC5zdG9wKCkpOwogIGJveC5yZW1vdmUoKTsKICBjb25zdCBidWYgPSBuZXcgVWludDhBcnJheShhd2FpdCBuZXcgQmxvYihjaHVua3MpLmFycmF5QnVmZmVyKCkpOwogIGxldCBzID0gJyc7IGZvciAobGV0IGkgPSAwOyBpIDwgYnVmLmxlbmd0aDsgaSArPSA4MTkyKSBzICs9IFN0cmluZy5mcm9tQ2hhckNvZGUoLi4uYnVmLnN1YmFycmF5KGksIGkgKyA4MTkyKSk7CiAgcmV0dXJuIGJ0b2Eocyk7Cn0KIiIiCgoKZGVmIHJlY29yZF9xdWVzdGlvbihzZWNvbmRzOiBpbnQgPSA2KSAtPiBzdHI6CiAgICAiIiJSZWNvcmQgZnJvbSBZT1VSIGJyb3dzZXIncyBtaWNyb3Bob25lIChDb2xhYiBvbmx5KS4gUmV0dXJucyB0aGUgc2F2ZWQgZmlsZSBwYXRoLiIiIgogICAgZnJvbSBnb29nbGUuY29sYWIub3V0cHV0IGltcG9ydCBldmFsX2pzCiAgICBmcm9tIElQeXRob24uZGlzcGxheSBpbXBvcnQgSmF2YXNjcmlwdCwgZGlzcGxheQogICAgZGlzcGxheShKYXZhc2NyaXB0KFJFQ09SRF9KUykpCiAgICBiNjQgPSBldmFsX2pzKGYicmVjb3JkUXVlc3Rpb24oe2ludChzZWNvbmRzICogMTAwMCl9KSIpCiAgICBwYXRoID0gIi9jb250ZW50L3F1ZXN0aW9uLndlYm0iCiAgICBQYXRoKHBhdGgpLndyaXRlX2J5dGVzKGJhc2U2NC5iNjRkZWNvZGUoYjY0KSkKICAgIHJldHVybiBwYXRoCg=="
}
proj = pathlib.Path("/content/third_eye"); proj.mkdir(exist_ok=True)
for name, data in FILES.items():
    (proj / name).write_bytes(base64.b64decode(data))
if str(proj) not in sys.path:
    sys.path.insert(0, str(proj))
print("✅ Setup done")

## 2. Settings

In [ ]:
#@title ⚙️ Settings: fill in, then press ▶ { display-mode: "form" }
COURSE = "BIOL 1010"  #@param {type:"string"}
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
WHISPER_MODEL = "base.en"  #@param ["tiny.en", "base.en", "small.en", "medium.en"]
#@markdown `tiny.en` = fastest, `small.en` = more accurate. `base.en` is a good middle.

import importlib, pathlib
import config, sync, assistant, study_sheet, colab_helpers
for m in (config, sync, assistant, study_sheet, colab_helpers):
    importlib.reload(m)

# API keys from Colab Secrets (optional): GEMINI_API_KEY (free tier) or ANTHROPIC_API_KEY
from google.colab import userdata
def _secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""
config.ANTHROPIC_API_KEY = _secret("ANTHROPIC_API_KEY")
config.GEMINI_API_KEY = _secret("GEMINI_API_KEY")
print("🤖 AI:", (config.ai_provider() + " API") if config.ai_provider() else "offline mode (no API key secret)")

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    folder = pathlib.Path("/content/drive/MyDrive/ThirdEye")
else:
    folder = pathlib.Path("/content/ThirdEyeNotes")

writer = sync.NotesWriter(COURSE, folder)
print("📁 Notes folder:", writer.folder)

## 3. Upload the lecture
An audio file (mp3, m4a, wav) or a video (mp4, mov). Use a **video** if you want slides captured too.

*No recording handy? Ask your professor for one, or record a 2–3 minute clip of a YouTube lecture with your phone.*

In [ ]:
#@title ⬆️ Upload a lecture recording { display-mode: "form" }
from google.colab import files
uploaded = files.upload()
LECTURE = "/content/" + next(iter(uploaded))
pathlib.Path(LECTURE).write_bytes(uploaded[next(iter(uploaded))])
print("Got it:", LECTURE)

## 4. Speech to text 🎙️
A 1-hour lecture takes about 3–6 minutes on a GPU and roughly 15–30 minutes on CPU with `base.en`. Lines appear as they're transcribed.

In [ ]:
#@title ▶ Transcribe the lecture { display-mode: "form" }
USE_GPU = True  #@param {type:"boolean"}
#@markdown If the notebook crashes or restarts during this step, untick `USE_GPU` and run it again.
writer.entries = [e for e in writer.entries if e[1] != "speech"]   # safe to re-run
colab_helpers.transcribe_file(LECTURE, writer, WHISPER_MODEL, use_gpu=USE_GPU and colab_helpers._gpu_available())

## 5. Third Eye camera: capture slides 👁️ *(optional)*
If your lecture is a video, this finds every new slide or board and reads its text. You can also upload photos of slides or the whiteboard you took yourself.

In [ ]:
#@title ▶ Capture slides { display-mode: "form" }
SCAN_VIDEO_FOR_SLIDES = True  #@param {type:"boolean"}
ALSO_UPLOAD_PHOTOS = False  #@param {type:"boolean"}
if SCAN_VIDEO_FOR_SLIDES:
    colab_helpers.extract_slides(LECTURE, writer, every_sec=1.0)
if ALSO_UPLOAD_PHOTOS:
    print("Choose photos of slides / whiteboard:")
    photos = files.upload()
    paths = []
    for name, data in photos.items():
        p = pathlib.Path("/content") / name; p.write_bytes(data); paths.append(p)
    colab_helpers.add_slide_photos(paths, writer)

## 6. Ask the assistant 💬
It answers from the lecture (and the slides) in 1–3 sentences and reads the answer out loud. Run the cell again for each new question.

In [ ]:
#@title ⌨️ Type a question { display-mode: "form" }
QUESTION = "What is the final electron acceptor?"  #@param {type:"string"}
colab_helpers.ask(QUESTION, writer)

In [ ]:
#@title 🎤 Or ask out loud (uses your browser's microphone; allow access when asked) { display-mode: "form" }
SECONDS = 6  #@param {type:"slider", min:3, max:15, step:1}
clip = colab_helpers.record_question(SECONDS)
q = colab_helpers.transcribe_clip(clip, WHISPER_MODEL)
if q:
    colab_helpers.ask(q, writer)
else:
    print("Didn't catch that. Run the cell again and speak a bit louder.")

## 7. Study sheet + reminders 📝⏰

In [ ]:
#@title ▶ Make the study sheet { display-mode: "form" }
from IPython.display import Markdown, display
out = study_sheet.generate(writer)
display(Markdown(out["md"].read_text()))
print("\nSaved in:", writer.folder)
if SAVE_TO_GOOGLE_DRIVE:
    print("📱 On your phone: open the Google Drive app → My Drive → ThirdEye.")
    print("   Tap reminders.ics to add the review reminders to your calendar.")

In [ ]:
#@title ⬇️ Download the files to this computer { display-mode: "form" }
for key in ("docx", "md", "ics"):
    if key in out:
        files.download(str(out[key]))
files.download(str(writer.notes_path))

## 8. Full notes *(optional)*
The complete time-stamped transcript, with slide text and your questions in place.

In [ ]:
#@title 📄 Show the full notes { display-mode: "form" }
display(Markdown(writer.notes_path.read_text()))